# An Explainable Deep Learning Framework for Ordinal Multi-Class Severity Grading of Androgenetic Alopecia

**CSE 463 — Department of Computer Science & Engineering, BRAC University**
Mahibi Islam

---

## What this notebook does

It implements the full pipeline from the project proposal, in the order the proposal demands —
**audit first, train second, evaluate once** — on the `hair_norwood_hamilton` Roboflow dataset
(CC BY 4.0, https://universe.roboflow.com/onuralpszr/hair_norwood_hamilton).

| § | Stage | Output |
|---|---|---|
| 1 | Data acquisition | dataset unpacked, manifest built |
| 2 | **Data audit** | leakage / duplicate / shortcut report + colour-only control |
| 3 | Canonical dataset | one image per leak-free group |
| 4 | Splits | locked test set + grouped CV folds, sha256-locked |
| 5–6 | Pipeline & models | ResNet-50 + CE / **CORAL** / **CORN** ordinal heads |
| 7–8 | Metrics & training | QWK, MAE, macro-F1, balanced acc., bootstrap CIs |
| 9–10 | Baselines & benchmark | colour-only, small CNN, SVM, ResNet variants |
| 11 | Locked-test evaluation | evaluated **once**, with calibration |
| 12 | Grad-CAM | heatmaps + a quantitative shortcut-attention metric |
| 13 | **Leakage ablation** | the headline result: vendor split vs. audited split |
| 14 | Model card | reproducibility artefact |

---

## ⚠️ Read this before you run anything

A full audit of this dataset was carried out before the notebook was written. **The vendor's
train/valid/test split is not usable as published.** Four independent problems were found; §2
re-derives every number below from scratch so you can verify them yourself.

**Finding 1 — 53% of the test set leaks from train.**
The 5,783 files are not 5,783 distinct photographs. They are ~1,450 source photographs, each
present up to 20 times as colour-augmented copies (`_gauss`, `_sepia`, `_rgbshift`,
`_channeldrop`) that Roboflow scattered across train/valid/test. 132 of 248 test images (53.2%)
have a near-duplicate (perceptual-hash Hamming ≤ 5) sitting in train. Byte-level deduplication
does not catch this, because every copy is a distinct JPEG.

**Finding 2 — the augmentation itself predicts the label.**
Stage 1 is **100% un-augmented**; stage 2 is 92.6% un-augmented; stages 3–7 are each ~80%
augmented. So "does this image have a green channel-drop tint?" is very nearly the decision rule
*"is this stage ≥ 3?"* — a signal with no clinical content whatsoever.

**Finding 3 — the source collection also predicts the label.**
Filenames come from three collections. `eu-<uuid>` and `exhibition_*` contribute **zero** stage-1
images and supply 58% of all stage-7 images. Camera, lighting and background are therefore
correlated with severity.

**Finding 4 — the effective sample size is ~4× smaller than it looks.**
After grouping by source photograph, there are **1,447 independent images**, not 5,783 — and the
class imbalance is far worse than the file counts suggest: **stage 6 has only 19 independent
images** (38:1 imbalance vs. 7:1 at file level).

**The control experiment that settles it (§2.11).** A logistic regression on **28 colour
statistics only** — no shape, no texture, no scalp structure — scores **QWK 0.715** and **82.3%**
on "stage ≤2 vs ≥3" on the vendor split. On the audited split the same features fall to QWK 0.431
with accuracy *below* the majority-class baseline.

> **Therefore: QWK 0.715 is the floor a vendor-split number has to clear before it means anything.**
> Any paper reporting ~90% accuracy on this dataset's native split is, on this evidence, reporting
> the colour shortcut.

**The human ceiling.** Inter-observer reliability of the Hamilton–Norwood scale among
dermatologists and residents is only **ICC 0.63–0.68** (Guarrera et al., *Int. J. Trichology*, 2009).
A model scoring κ ≈ 0.65–0.70 is already at expert-agreement level. A model scoring 0.95 is
reading an artefact, not a scalp.

---

*Research prototype. Not a diagnostic device. See §14.*


## 0 · Environment

Runtime → Change runtime type → **T4 GPU** (or better). Everything below is Keras-3 / TF-2.16+
compatible and also runs on Keras 2.


In [ ]:
import sys, subprocess, platform
print("python      :", sys.version.split()[0], "|", platform.platform())
import tensorflow as tf
print("tensorflow  :", tf.__version__)
try:
    import keras; print("keras       :", keras.__version__)
except Exception:
    keras = tf.keras; print("keras       : bundled with tf")
gpus = tf.config.list_physical_devices("GPU")
print("GPUs        :", gpus or "NONE  <-- switch to a GPU runtime, training will be very slow")
if gpus:
    try: print("gpu name    :", tf.config.experimental.get_device_details(gpus[0]).get("device_name", "?"))
    except Exception: pass
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null || true


In [ ]:
# ---------------------------------------------------------------------------------
# CONFIG  — everything tunable lives here.
# ---------------------------------------------------------------------------------
from types import SimpleNamespace

CFG = SimpleNamespace(
    # --- reproducibility -----------------------------------------------------------
    SEED                    = 1337,

    # --- audit ---------------------------------------------------------------------
    PHASH_THRESHOLD         = 5,      # Hamming distance on a 64-bit pHash. Justified in 2.6.
    EXCLUDE_AMBIGUOUS       = True,   # drop groups whose own copies carry conflicting labels
                                      # (proposal: "ambiguous examples are excluded")

    # --- dataset construction -------------------------------------------------------
    IMG_SIZE                = 224,    # proposal: 224x224
    N_CLASSES               = 7,      # 7 = Norwood I..VII (primary). 5 = collapse {5,6,7} (secondary)
    ONE_IMAGE_PER_GROUP     = True,   # canonical protocol. False keeps near-dup copies (grouped).

    # --- splits ---------------------------------------------------------------------
    TEST_FRACTION           = 0.20,   # locked test set, never touched until 11
    N_FOLDS                 = 5,      # grouped stratified CV on the development set
    N_REPEATS               = 1,      # raise to 3-5 for tighter CIs (multiplies runtime)

    # --- training ---------------------------------------------------------------------
    BATCH_SIZE              = 32,
    EPOCHS_HEAD             = 15,     # stage A: frozen backbone
    EPOCHS_FINETUNE         = 35,     # stage B: unfrozen from FINETUNE_FROM
    LR_HEAD                 = 1e-3,
    LR_FINETUNE             = 1e-5,
    FINETUNE_FROM           = "conv4_block1_1_conv",   # unfreeze this layer onward
    FREEZE_BN_ON_FINETUNE   = True,   # standard for small medical sets
    EARLY_STOP_PATIENCE     = 10,
    MONITOR                 = "val_qwk",   # ordinal-aware model selection, not val_accuracy
    LABEL_SMOOTHING         = 0.05,
    USE_CLASS_WEIGHTS       = True,

    # --- evaluation -------------------------------------------------------------------
    N_BOOTSTRAP             = 2000,

    # --- convenience ------------------------------------------------------------------
    QUICK_RUN               = True,   # <-- True: 1 fold, few epochs, ~10 min. Set False for the real run.
    RUN_LEAKAGE_ABLATION    = True,   # 13 — the headline experiment
    OUT_DIR                 = "/content/outputs",
    DRIVE_BACKUP            = None,   # e.g. "/content/drive/MyDrive/CSE463" to copy artefacts out
)

if CFG.QUICK_RUN:
    CFG.EPOCHS_HEAD, CFG.EPOCHS_FINETUNE = 3, 5
    CFG.N_FOLDS, CFG.N_REPEATS, CFG.N_BOOTSTRAP = 2, 1, 400
    print(">>> QUICK_RUN active: results are a smoke test, NOT the numbers you report.")

import os, random, numpy as np
if not os.path.isdir("/content"):      # not on Colab: write next to the repo instead
    CFG.OUT_DIR = "../results"
os.makedirs(CFG.OUT_DIR, exist_ok=True)
os.makedirs(f"{CFG.OUT_DIR}/figures", exist_ok=True)

def set_seed(s=CFG.SEED):
    os.environ["PYTHONHASHSEED"] = str(s)
    random.seed(s); np.random.seed(s); tf.random.set_seed(s)
    try: tf.keras.utils.set_random_seed(s)
    except Exception: pass
set_seed()

# crisp inline figures in Colab; no-ops outside IPython so the notebook stays scriptable
try:
    _ip = get_ipython()
    _ip.run_line_magic("matplotlib", "inline")
    _ip.run_line_magic("config", "InlineBackend.figure_format = 'retina'")
except Exception:
    pass

import matplotlib; matplotlib.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 200, "font.size": 9,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
    "grid.alpha": .25, "figure.facecolor": "white", "savefig.bbox": "tight"})
import matplotlib.pyplot as plt
STAGE_COLORS = ["#2a9d8f", "#57b8a0", "#e9c46a", "#f4a261", "#ee8959", "#e76f51", "#bc4749"]

FIGURES = {}          # name -> path, so 15 can rebuild the contact sheet in order
def savefig(name):
    path = f"{CFG.OUT_DIR}/figures/{name}.png"
    plt.savefig(path); FIGURES[name] = path
    plt.show()            # render inline in Colab as well as writing the file
    plt.close("all")

print("config ready | seed", CFG.SEED, "| outputs ->", CFG.OUT_DIR)


## 1 · Data acquisition

Pick **one** route. Route A (Google Drive) is the most reliable — upload
`hair_norwood_hamilton.v1i.multiclass.zip` to your Drive once and never re-upload it.


In [ ]:
import zipfile, shutil, glob
from pathlib import Path

ZIP_NAME = "hair_norwood_hamilton.v1i.multiclass.zip"
DATA_ROOT = Path("/content/data")

# ---- Route A: Google Drive (recommended) --------------------------------------------
USE_DRIVE = True
DRIVE_ZIP = f"/content/drive/MyDrive/{ZIP_NAME}"   # <-- adjust to wherever you put it

# ---- Route B: direct browser upload --------------------------------------------------
# set USE_DRIVE = False; a file picker appears (slow for 228 MB, but works)

# ---- Route D: local clone of the repository ------------------------------------------
# If the unzipped export sits in ../data/raw (see data/README.md), it is used directly and
# routes A-C are skipped. Run the notebook from the notebooks/ directory.
LOCAL_RAW = Path("../data/raw")

# ---- Route C: Roboflow API -----------------------------------------------------------
#   !pip -q install roboflow
#   from roboflow import Roboflow
#   rf = Roboflow(api_key="YOUR_KEY")
#   rf.workspace("onuralpszr").project("hair_norwood_hamilton").version(1).download("multiclass")

zip_path = None
if (LOCAL_RAW / "train" / "_classes.csv").exists():
    DATA_ROOT = LOCAL_RAW.resolve()
    print("using local dataset:", DATA_ROOT)
elif USE_DRIVE:
    from google.colab import drive
    if not os.path.ismount("/content/drive"): drive.mount("/content/drive")
    assert os.path.exists(DRIVE_ZIP), f"not found: {DRIVE_ZIP}\nUpload the zip to Drive or switch to Route B."
    zip_path = DRIVE_ZIP
else:
    from google.colab import files
    up = files.upload()
    zip_path = "/content/" + list(up.keys())[0]

if zip_path and not (DATA_ROOT / "train" / "_classes.csv").exists():
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    print("extracting ...")
    with zipfile.ZipFile(zip_path) as z: z.extractall(DATA_ROOT)
print("dataset root:", DATA_ROOT)
for p in sorted(DATA_ROOT.iterdir())[:8]: print("  ", p.name)
print(Path(DATA_ROOT / "README.dataset.txt").read_text())


## 2 · Data audit

> *"A rigorous audit of the training dataset to catch duplicate images, mislabeling, and data
> leakage **before any training happens**."* — project proposal, §2

Nothing in §5+ is allowed to run until this section has produced its verdict. Every claim in the
header is re-derived here from the raw files.

### 2.1 · Manifest and label integrity


In [ ]:
import pandas as pd, csv, hashlib, collections, re, io, glob, itertools, time
from pathlib import Path
from PIL import Image

SPLITS = ["train", "valid", "test"]

rows = []
for split in SPLITS:
    df = pd.read_csv(DATA_ROOT / split / "_classes.csv", skipinitialspace=True)
    df.columns = [c.strip() for c in df.columns]
    cls_cols = [c for c in df.columns if c != "filename"]
    for _, r in df.iterrows():
        onehot = r[cls_cols].astype(int).values
        rows.append(dict(vendor_split=split, filename=str(r["filename"]).strip(),
                         n_pos=int(onehot.sum()),
                         stage=int(np.argmax(onehot)) + 1 if onehot.sum() == 1 else -1))
man = pd.DataFrame(rows)
man["path"] = man.apply(lambda r: str(DATA_ROOT / r.vendor_split / r.filename), axis=1)

print("declared class columns :", cls_cols)
print("total rows             :", len(man))
print("README claims 5783 imgs:", len(man) == 5783)
print()
print("LABEL INTEGRITY")
print("  rows with exactly one positive class :", (man.n_pos == 1).sum())
print("  UNLABELED (all zero)                 :", (man.n_pos == 0).sum())
print("  MULTI-LABEL (>1 positive)            :", (man.n_pos > 1).sum())
missing = [p for p in man.path if not os.path.exists(p)]
print("  manifest rows with no file on disk   :", len(missing))
on_disk = {os.path.basename(p) for p in glob.glob(str(DATA_ROOT / "*" / "*.jpg"))}
print("  image files on disk not in manifest  :", len(on_disk - set(man.filename)))
assert (man.n_pos == 1).all() and not missing, "label/file integrity failed"
man = man.drop(columns=["n_pos"])

print("\nVENDOR SPLIT x STAGE (file counts)")
ct = pd.crosstab(man.stage, man.vendor_split).reindex(columns=SPLITS)
ct["TOTAL"] = ct.sum(1); display(ct)


### 2.2 · Image property audit


In [ ]:
sizes, modes, corrupt = collections.Counter(), collections.Counter(), []
t0 = time.time()
for i, p in enumerate(man.path):
    try:
        with Image.open(p) as im:
            sizes[im.size] += 1; modes[im.mode] += 1
            im.verify()
    except Exception as e:
        corrupt.append((p, str(e)))
    if (i + 1) % 2000 == 0: print(f"   {i+1}/{len(man)}")
print(f"\nchecked {len(man)} files in {time.time()-t0:.0f}s")
print("resolutions :", dict(sizes))
print("colour modes:", dict(modes))
print("corrupt     :", len(corrupt))
print("\nNOTE: every image is already 640x640 RGB. Roboflow applied auto-orientation (EXIF stripped)")
print("and a STRETCH resize to 640x640 -- aspect ratio is therefore already distorted and the")
print("original framing is unrecoverable. Our 224x224 resize inherits that distortion.")


### 2.3 · Filename decomposition

Roboflow filenames are `<source_id>[_<augmentation>]_jpg.rf.<content_hash>.jpg`. The `<source_id>`
identifies the **original photograph**; the optional `<augmentation>` token marks a colour variant
baked in by the dataset author *before* upload. Both matter enormously.


In [ ]:
KNOWN_AUG = {"channeldrop","gauss","rgbshift","sepia","clahe","blur","noise","bright","brightness",
             "contrast","hue","tone","equalize","jpegcomp","downscale","fog","shadow","solarize",
             "posterize","emboss","sharpen","elastic","grid","coarse","cutout","flip","rot90"}
RF_RE = re.compile(r"^(?P<stem>.+?)_jpg\.rf\.[0-9a-f]+\.jpg$")

def decompose(fn):
    m = RF_RE.match(fn)
    stem = m.group("stem") if m else fn
    aug = "original"
    tok = stem.rsplit("_", 1)
    if len(tok) == 2 and tok[1] in KNOWN_AUG:
        aug, stem = tok[1], tok[0]
    return stem, aug

def source_family(stem):
    if stem.startswith("exhibition_"): return "exhibition_*"
    if stem.startswith("eu-"):         return "eu-<uuid>"
    if re.match(r"^\d{14}", stem):     return "timestamp14"
    return "other"

man[["source_id", "vendor_aug"]] = man.filename.apply(lambda f: pd.Series(decompose(f)))
man["family"] = man.source_id.map(source_family)

# sanity: show every trailing token we saw, so nothing is silently misparsed
tails = collections.Counter(RF_RE.match(f).group("stem").rsplit("_", 1)[-1]
                            for f in man.filename if RF_RE.match(f) and "_" in RF_RE.match(f).group("stem"))
print("trailing tokens seen (top 15) -- anything augmentation-like MUST be in KNOWN_AUG:")
for k, v in tails.most_common(15): print(f"   {k:<16} {v:>6}  {'<-- AUG' if k in KNOWN_AUG else ''}")
print()
print("augmentation variants :", dict(man.vendor_aug.value_counts()))
print("source families       :", dict(man.family.value_counts()))
print("distinct source photos:", man.source_id.nunique(), f"  (vs {len(man)} files = {len(man)/man.source_id.nunique():.1f}x replication)")


### 2.4 · Exact duplicates (MD5)


In [ ]:
md5 = []
for i, p in enumerate(man.path):
    with open(p, "rb") as f: md5.append(hashlib.md5(f.read()).hexdigest())
    if (i + 1) % 2000 == 0: print(f"   {i+1}/{len(man)}")
man["md5"] = md5
dups = man.groupby("md5").filter(lambda g: len(g) > 1)
print(f"\nbyte-identical duplicate groups : {dups.md5.nunique()}")
print(f"redundant copies                : {len(dups) - dups.md5.nunique()}")
print("\nRoboflow already removes byte-identical files, so this comes back clean.")
print("That is exactly why byte-level dedup is NOT sufficient here -- see 2.5.")


### 2.5 · Perceptual near-duplicates (pHash)

Every augmented copy is a *different* JPEG but the *same photograph*. A 64-bit DCT perceptual hash
sees through the colour transforms.


In [ ]:
from scipy.fftpack import dct

def phash64(img):
    a = np.asarray(img.convert("L").resize((32, 32), Image.BILINEAR), dtype=np.float32)
    d = dct(dct(a, axis=0), axis=1)[:8, :8].flatten()
    return (d > np.median(np.delete(d, 0))).astype(np.uint8)

t0 = time.time(); bits = np.zeros((len(man), 64), np.uint8)
for i, p in enumerate(man.path):
    with Image.open(p) as im: bits[i] = phash64(im)
    if (i + 1) % 1000 == 0: print(f"   {i+1}/{len(man)}  ({time.time()-t0:.0f}s)")
print(f"hashed {len(man)} images in {time.time()-t0:.0f}s")

def near_dup_pairs(bits, thr, chunk=512):
    b = bits.astype(np.int16); out = []
    for i0 in range(0, len(b), chunk):
        d = (b[i0:i0+chunk, None, :] != b[None, :, :]).sum(2)
        ii, jj = np.where(d <= thr)
        for a, c in zip(ii, jj):
            if i0 + a < c: out.append((i0 + int(a), int(c), int(d[a, c])))
    return out

pairs = near_dup_pairs(bits, CFG.PHASH_THRESHOLD)
print(f"\nnear-duplicate pairs at Hamming <= {CFG.PHASH_THRESHOLD}: {len(pairs):,}")


### 2.6 · Choosing the pHash threshold

A threshold that is too tight misses augmented copies; too loose and it starts merging genuinely
different patients. We sweep it and pick a value inside the stable plateau.


In [ ]:
class DSU:
    def __init__(s, n): s.p = list(range(n))
    def find(s, x):
        while s.p[x] != x: s.p[x] = s.p[s.p[x]]; x = s.p[x]
        return x
    def union(s, a, b):
        ra, rb = s.find(a), s.find(b)
        if ra != rb: s.p[rb] = ra

src_index = collections.defaultdict(list)
for i, sid in enumerate(man.source_id): src_index[sid].append(i)

def build_groups(thr):
    d = DSU(len(man))
    for v in src_index.values():          # same source photograph -> same group, always
        for x in v[1:]: d.union(v[0], x)
    for a, b, _ in near_dup_pairs(bits, thr): d.union(a, b)   # plus perceptual links
    return np.array([d.find(i) for i in range(len(man))])

sweep = []
for thr in [0, 3, 5, 8, 10, 12]:
    g = build_groups(thr); c = collections.Counter(g)
    sweep.append(dict(threshold=thr, n_groups=len(c), largest_group=max(c.values())))
sweep = pd.DataFrame(sweep); display(sweep)

fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].plot(sweep.threshold, sweep.n_groups, "o-", color="#264653")
ax[0].axvline(CFG.PHASH_THRESHOLD, ls="--", c="#e76f51"); ax[0].set(xlabel="pHash Hamming threshold", ylabel="# independent groups")
ax[1].semilogy(sweep.threshold, sweep.largest_group, "o-", color="#264653")
ax[1].axvline(CFG.PHASH_THRESHOLD, ls="--", c="#e76f51"); ax[1].set(xlabel="pHash Hamming threshold", ylabel="largest group (log)")
fig.suptitle("Grouping stability — chosen threshold marked", y=1.02); savefig("02_phash_threshold")

print(f"""
Groups are flat at ~1449 for thr 0-3 and 1447 at thr=5, then collapse: at thr=10 the largest
group swallows hundreds of images, i.e. distinct patients are being merged. thr={CFG.PHASH_THRESHOLD}
sits in the plateau -- conservative enough not to over-merge, loose enough to catch every
colour-augmented copy. Note it only merges a handful of source_ids beyond the filename grouping,
so the result barely depends on this choice.""")

man["group"] = build_groups(CFG.PHASH_THRESHOLD)
print("independent groups:", man.group.nunique())


### 2.7 · Leakage in the vendor split

**Finding 1.** How much of the published test set is already in train?


In [ ]:
g2s = man.groupby("group").vendor_split.nunique()
straddling = set(g2s[g2s > 1].index)
print(f"groups straddling >1 vendor split : {len(straddling)} / {man.group.nunique()}")

leak = []
for tgt in ["test", "valid"]:
    sub = man[man.vendor_split == tgt]
    tr_groups = set(man[man.vendor_split == "train"].group)
    n = sub.group.isin(tr_groups).sum()
    leak.append(dict(split=tgt, n=len(sub), leaked=int(n), pct=100 * n / len(sub)))
leak = pd.DataFrame(leak); display(leak)

sp = man.vendor_split.values
xpair = [(a, b) for a, b, _ in pairs if sp[a] != sp[b]]
print(f"near-duplicate pairs crossing split boundaries : {len(xpair):,}")
print(f"  train<->test  : {sum(1 for a,b in xpair if {sp[a],sp[b]}=={'train','test'}):,}")
print(f"  train<->valid : {sum(1 for a,b in xpair if {sp[a],sp[b]}=={'train','valid'}):,}")

fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(leak.split, leak.pct, color=["#bc4749", "#e76f51"])
for i, r in leak.iterrows(): ax.text(i, r.pct + 1.5, f"{r.pct:.1f}%\n({r.leaked}/{r.n})", ha="center", fontsize=8)
ax.set(ylabel="% of split with a near-duplicate in train", ylim=(0, 72),
       title="Finding 1 — the vendor split leaks")
savefig("02_leakage")
print("\nVERDICT: over half of the published test set is a recoloured copy of a training image.")
print("Any metric computed on this split is measuring memorisation, not generalisation.")


### 2.8 · Shortcut #1 — the augmentation predicts the label

**Finding 2.** If the augmentation marker and the stage are correlated, the network can win by
detecting the colour filter rather than the scalp.


In [ ]:
aug_order = ["original"] + sorted(set(man.vendor_aug) - {"original"})
ct_aug = pd.crosstab(man.stage, man.vendor_aug).reindex(columns=aug_order, fill_value=0)
pct_aug = (100 * ct_aug.div(ct_aug.sum(1), axis=0)).round(1)
print("AUGMENTATION VARIANT x STAGE  (row %)"); display(pct_aug)

from scipy.stats import chi2_contingency
chi2, p, dof, _ = chi2_contingency(ct_aug.values)
cv = np.sqrt(chi2 / (ct_aug.values.sum() * (min(ct_aug.shape) - 1)))
print(f"chi2 = {chi2:,.0f}   p = {p:.3g}   Cramer's V = {cv:.3f}   (V>0.25 = strong association)")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
pct_aug.plot(kind="bar", stacked=True, ax=ax[0], colormap="Spectral", width=.8)
ax[0].set(xlabel="Norwood stage", ylabel="% of stage's files", title="Augmentation mix per stage")
ax[0].legend(fontsize=7, ncol=2); ax[0].tick_params(axis="x", rotation=0)
frac = 100 * (1 - ct_aug["original"] / ct_aug.sum(1))
ax[1].bar(frac.index, frac.values, color=STAGE_COLORS)
for i, v in zip(frac.index, frac.values): ax[1].text(i, v + 2, f"{v:.0f}%", ha="center", fontsize=8)
ax[1].set(xlabel="Norwood stage", ylabel="% augmented", ylim=(0, 100),
          title="Finding 2 — 'is it tinted?' ~ 'is it stage >= 3?'")
savefig("02_shortcut_augmentation")

print(f"""
Stage 1 is {frac[1]:.0f}% augmented and stage 2 is {frac[2]:.0f}%, while stages 3-7 sit near 80%.
A single binary feature -- presence of a colour filter -- therefore separates {{1,2}} from {{3..7}}
almost perfectly. 2.11 measures exactly how far that gets you.""")


### 2.9 · Shortcut #2 — the source collection predicts the label

**Finding 3.** Three distinct filename families are present. If severity correlates with the
collection, then camera, lighting, background and population correlate with severity too.


In [ ]:
ct_fam = pd.crosstab(man.family, man.stage)
print("SOURCE FAMILY x STAGE (file counts)"); display(ct_fam)
print("\n(row %)"); display((100 * ct_fam.div(ct_fam.sum(1), axis=0)).round(1))
chi2, p, dof, _ = chi2_contingency(ct_fam.values)
print(f"chi2 = {chi2:,.0f}   p = {p:.3g}   Cramer's V = {np.sqrt(chi2/(ct_fam.values.sum()*(min(ct_fam.shape)-1))):.3f}")

share = 100 * ct_fam.div(ct_fam.sum(0), axis=1)
fig, ax = plt.subplots(figsize=(6, 3.2))
share.T.plot(kind="bar", stacked=True, ax=ax, color=["#e76f51", "#f4a261", "#2a9d8f"], width=.8)
ax.set(xlabel="Norwood stage", ylabel="% of that stage's files", title="Finding 3 — source collection tracks severity")
ax.tick_params(axis="x", rotation=0); ax.legend(fontsize=8, title=None)
savefig("02_shortcut_source")
print("\nThe eu-<uuid> and exhibition_* collections contain NO stage-1 images at all and together")
print("supply about half of stages 6-7. Background and camera are confounded with the label.")
print("This is the specific failure Grad-CAM is supposed to expose (12.3).")


### 2.10 · Label consistency inside a group

Copies of the same photograph must carry the same label. Where they do not, the label is
demonstrably unreliable.


In [ ]:
# named aggregations -- do NOT call a column "nunique"/"count", those shadow DataFrame methods
gl = man.groupby("group").stage.agg(n_labels="nunique", lo="min", hi="max", n_files="count")
bad = gl[gl.n_labels > 1]
print(f"groups whose own copies disagree : {len(bad)}  (files affected: {int(bad.n_files.sum())})")
if len(bad):
    print("max label spread within a group  :", int((bad.hi - bad.lo).max()), "stage(s)")
    display(man[man.group.isin(bad.index)]
            .groupby("group").agg(stages=("stage", lambda s: sorted(set(s))),
                                  n_files=("stage", "size"),
                                  example=("source_id", "first")).head(10))
print(f"""
Only ~{len(bad)} groups disagree internally, so gross label corruption is not the problem here.
But every disagreement is an ADJACENT pair (2 vs 3) -- consistent with the known
inter-observer unreliability of the scale (ICC 0.63-0.68), not with random noise.
CFG.EXCLUDE_AMBIGUOUS={CFG.EXCLUDE_AMBIGUOUS} controls whether these groups are dropped.""")


### 2.11 · Control experiment — how far does colour alone get you?

This is the decisive test. We build a model that **cannot see hair**: 28 global colour statistics
from a 32×32 thumbnail (mean/std RGB, mean/std HSV, 8-bin hue and saturation histograms). No
shape, no texture, no scalp structure. Then we score it on the vendor split and on the audited
split.

Whatever this model scores on the vendor split is the **floor** — a number a real model must
clearly exceed before it has demonstrated anything clinical.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score, balanced_accuracy_score

def colour_features(paths):
    F = np.zeros((len(paths), 28), np.float32)
    for i, p in enumerate(paths):
        with Image.open(p) as im:
            im = im.convert("RGB").resize((32, 32))
            a = np.asarray(im, np.float32) / 255.
            h = np.asarray(im.convert("HSV"), np.float32) / 255.
        F[i] = np.concatenate([a.mean((0,1)), a.std((0,1)), h.mean((0,1)), h.std((0,1)),
                               np.histogram(h[:,:,0], 8, (0,1), density=True)[0],
                               np.histogram(h[:,:,1], 8, (0,1), density=True)[0]])
        if (i+1) % 1500 == 0: print(f"   {i+1}/{len(paths)}")
    return F

COLOUR_X = colour_features(man.path.tolist())
y_all = man.stage.values

def _clf(): return make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, class_weight="balanced"))
def _scores(yt, yp):
    return dict(accuracy=accuracy_score(yt, yp), macro_f1=f1_score(yt, yp, average="macro"),
                balanced_acc=balanced_accuracy_score(yt, yp),
                QWK=cohen_kappa_score(yt, yp, weights="quadratic"),
                MAE=float(np.abs(yt - yp).mean()),
                adjacent_acc=float((np.abs(yt - yp) <= 1).mean()))

tr, te = (man.vendor_split != "test").values, (man.vendor_split == "test").values
m = _clf().fit(COLOUR_X[tr], y_all[tr]); A = _scores(y_all[te], m.predict(COLOUR_X[te]))
yb = (y_all >= 3).astype(int)
mb = _clf().fit(COLOUR_X[tr], yb[tr]); A["binary_le2_vs_ge3"] = accuracy_score(yb[te], mb.predict(COLOUR_X[te]))
A["binary_chance"] = max(yb[te].mean(), 1 - yb[te].mean())
print("A) VENDOR SPLIT, all files      ", {k: round(v, 3) for k, v in A.items()})
COLOUR_VENDOR = A


### 2.12 · Canonical grouping and the audit verdict

Now the same colour-only model on the audited protocol: one image per independent group,
grouped-stratified CV.


In [ ]:
# one representative per group: prefer an un-augmented original, deterministic tie-break on filename
man["_is_original"] = (man.vendor_aug == "original").astype(int)
rep_idx = np.sort(man.sort_values(["group", "_is_original", "filename"], ascending=[True, False, True])
                     .groupby("group").head(1).index.values)
assert len(rep_idx) == man.group.nunique()

grp_stage = man.groupby("group").stage.agg(lambda s: s.mode().min())  # majority, ties -> lower stage
man["group_stage"] = man.group.map(grp_stage)

Xc, yc = COLOUR_X[rep_idx], man.loc[rep_idx, "group_stage"].values
_folds = []
for a, b in StratifiedKFold(5, shuffle=True, random_state=CFG.SEED).split(Xc, yc):
    _folds.append(_scores(yc[b], _clf().fit(Xc[a], yc[a]).predict(Xc[b])))
COLOUR_AUDITED = {k: float(np.mean([f[k] for f in _folds])) for k in _folds[0]}
maj = collections.Counter(yc).most_common(1)[0][1] / len(yc)
print("B) AUDITED SPLIT, 1 img/group   ", {k: round(v, 3) for k, v in COLOUR_AUDITED.items()})
print(f"   majority-class baseline accuracy = {maj:.3f}")

cmp = pd.DataFrame([{"protocol": "vendor split (as published)", **{k: COLOUR_VENDOR[k] for k in ["accuracy","macro_f1","QWK"]}},
                    {"protocol": "audited split (1 img/group)", **{k: COLOUR_AUDITED[k] for k in ["accuracy","macro_f1","QWK"]}}]).set_index("protocol").round(3)
print("\nCOLOUR-ONLY MODEL — no shape, no texture, no scalp information:"); display(cmp)

fig, ax = plt.subplots(figsize=(5.5, 3))
w = .35; x = np.arange(3); k = ["accuracy", "macro_f1", "QWK"]
ax.bar(x-w/2, [COLOUR_VENDOR[i] for i in k], w, label="vendor split", color="#bc4749")
ax.bar(x+w/2, [COLOUR_AUDITED[i] for i in k], w, label="audited split", color="#2a9d8f")
ax.axhline(maj, ls=":", c="k", lw=1); ax.text(2.3, maj+.01, "majority baseline", fontsize=7)
ax.set_xticks(x, ["accuracy", "macro-F1", "QWK"]); ax.legend(fontsize=8)
ax.set(title="Colour statistics alone — the shortcut, measured", ylim=(0, 1))
savefig("02_colour_control")

print(f"""
================================  AUDIT VERDICT  ================================
 files in dataset ............ {len(man):,}
 distinct source photographs .. {man.source_id.nunique():,}
 INDEPENDENT groups ........... {man.group.nunique():,}   <-- the real sample size
 byte-identical duplicates .... 0     (Roboflow dedups; useless here)
 test images leaked from train  {leak.loc[0,'leaked']}/{leak.loc[0,'n']} ({leak.loc[0,'pct']:.1f}%)
 aug-vs-stage Cramer's V ...... strong      (stage 1 = 0% augmented, stages 3-7 ~80%)
 source-vs-stage confound ..... strong      (eu-/exhibition contribute no stage-1 at all)
 groups w/ conflicting labels . {len(bad)}

 COLOUR-ONLY FLOOR on vendor split : QWK {COLOUR_VENDOR['QWK']:.3f} | {COLOUR_VENDOR['binary_le2_vs_ge3']:.1%} on 'stage<=2 vs >=3' (chance {COLOUR_VENDOR['binary_chance']:.1%})
 COLOUR-ONLY on audited split      : QWK {COLOUR_AUDITED['QWK']:.3f} | accuracy {COLOUR_AUDITED['accuracy']:.3f} (BELOW the {maj:.3f} majority baseline)

 DECISION: the vendor split is discarded. All modelling below uses the audited,
 group-disjoint protocol. The vendor split is retained for ONE purpose only --
 section 13, where it is used to measure how much it inflates results.
=================================================================================""")
man.to_csv(f"{CFG.OUT_DIR}/audit_manifest.csv", index=False, lineterminator="\n")
print("full manifest ->", f"{CFG.OUT_DIR}/audit_manifest.csv")


## 3 · Canonical dataset

One image per independent group. This is the dataset every model below is trained and evaluated
on.


In [ ]:
ds = man.loc[rep_idx, ["path", "filename", "source_id", "group", "group_stage", "family", "vendor_aug", "vendor_split"]].copy()
ds["man_idx"] = ds.index.values                  # keep the row id in `man` so COLOUR_X stays aligned
ds = ds.rename(columns={"group_stage": "stage"}).reset_index(drop=True)

if CFG.EXCLUDE_AMBIGUOUS and len(bad):
    n0 = len(ds); ds = ds[~ds.group.isin(bad.index)].reset_index(drop=True)
    print(f"excluded {n0-len(ds)} ambiguous groups (internally conflicting labels), per proposal 4")

if CFG.N_CLASSES == 5:
    ds["stage"] = ds.stage.clip(upper=5)     # {5,6,7} -> 5, ordinal structure preserved
    print("collapsed to 5 classes: 1,2,3,4,>=5")

ds["y"] = ds.stage - 1                        # 0-indexed ordinal target
K = int(ds.y.max()) + 1
CLASS_NAMES = [f"Stage {i+1}" for i in range(K)] if K == 7 else [f"Stage {i+1}" for i in range(K-1)] + [f"Stage >={K}"]

print(f"\nCANONICAL DATASET : {len(ds)} independent images, {K} ordinal classes")
dist = ds.stage.value_counts().sort_index()
tab = pd.DataFrame({"n_images": dist, "% of set": (100*dist/len(ds)).round(1),
                    "n_files_in_vendor_set": man.groupby("stage").size().reindex(dist.index)})
tab.index.name = "Norwood stage"; display(tab)
print(f"imbalance ratio (max/min) : {dist.max()/dist.min():.1f} : 1"
      f"   [at FILE level it merely looks like {man.stage.value_counts().max()/man.stage.value_counts().min():.1f} : 1]")

fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
ax[0].bar(dist.index, dist.values, color=STAGE_COLORS[:len(dist)])
for i, v in zip(dist.index, dist.values): ax[0].text(i, v+8, str(v), ha="center", fontsize=8)
ax[0].set(xlabel="Norwood stage", ylabel="independent images", title=f"Audited dataset (n={len(ds)})")
fc = man.stage.value_counts().sort_index()
ax[1].bar(fc.index-.2, fc.values, .4, label="vendor files", color="#bc4749")
ax[1].bar(dist.index+.2, dist.values, .4, label="independent images", color="#2a9d8f")
ax[1].set(xlabel="Norwood stage", ylabel="count", title="Replication hides the true imbalance"); ax[1].legend(fontsize=8)
savefig("03_class_distribution")

print(f"""
STATISTICAL POWER WARNING
  Stage {dist.idxmin()} has only {dist.min()} independent images in the ENTIRE dataset. With a {CFG.TEST_FRACTION:.0%}
  locked test set that is ~{int(dist.min()*CFG.TEST_FRACTION)} test images, so its per-class recall has a 95% CI
  roughly +/-{100*1.96*np.sqrt(.25/max(1,int(dist.min()*CFG.TEST_FRACTION))):.0f} points -- effectively uninformative on its own.
  This is why 10 reports POOLED cross-validated metrics as the primary result and treats the
  single locked-test number as a confirmatory check, not the headline.""")


## 4 · Splits

*Locked test set* (never touched before §11) + *k-fold grouped stratified CV* on the remainder.
Because §3 already reduced the data to one image per group, group-disjointness is guaranteed by
construction; we still carry `group` so §13 can add near-duplicate copies back into training only.

The split manifest is written to disk and fingerprinted with SHA-256 — that fingerprint is what
makes "the test set was locked" a checkable claim rather than a promise.


In [ ]:
from sklearn.model_selection import StratifiedKFold, RepeatedStratifiedKFold

y_ds = ds.y.values
sss = StratifiedKFold(int(round(1/CFG.TEST_FRACTION)), shuffle=True, random_state=CFG.SEED)
dev_idx, test_idx = next(iter(sss.split(np.zeros(len(ds)), y_ds)))
ds["role"] = "dev"; ds.loc[test_idx, "role"] = "locked_test"

folds = []
rskf = RepeatedStratifiedKFold(n_splits=CFG.N_FOLDS, n_repeats=CFG.N_REPEATS, random_state=CFG.SEED)
for f, (a, b) in enumerate(rskf.split(np.zeros(len(dev_idx)), y_ds[dev_idx])):
    folds.append((dev_idx[a], dev_idx[b]))
print(f"locked test : {len(test_idx)} images")
print(f"development : {len(dev_idx)} images -> {len(folds)} CV folds ({CFG.N_FOLDS} x {CFG.N_REPEATS})")

chk = pd.DataFrame({"dev": pd.Series(y_ds[dev_idx]+1).value_counts().sort_index(),
                    "locked_test": pd.Series(y_ds[test_idx]+1).value_counts().sort_index()}).fillna(0).astype(int)
chk.index.name = "stage"; display(chk)
assert set(ds.loc[dev_idx, "group"]).isdisjoint(set(ds.loc[test_idx, "group"])), "GROUP LEAK between dev and test"
print("group-disjointness dev/test : VERIFIED")

split_csv = f"{CFG.OUT_DIR}/split_manifest.csv"
ds[["filename", "source_id", "group", "stage", "y", "role"]].sort_values("filename").to_csv(split_csv, index=False, lineterminator="\n")
LOCK = hashlib.sha256(open(split_csv, "rb").read()).hexdigest()
print(f"\nsplit manifest -> {split_csv}")
print(f"SHA-256 LOCK   : {LOCK}")
print("Quote this hash in the paper. If it changes, the test set was re-drawn and the")
print("'evaluated once' claim no longer holds.")


## 5 · Input pipeline

1,447 images at 224×224×3 uint8 is ~218 MB — it all fits in RAM, so we decode once and train from
an in-memory array. That removes I/O from the loop entirely and makes the CV sweep cheap.

Augmentation lives **inside the Keras model** as preprocessing layers. That is not a stylistic
choice: Keras augmentation layers are active in `training=True` and are exact no-ops at inference,
which makes *"augmentation applied only to training data, never to validation/test"* a structural
guarantee rather than something we have to remember.

Colour augmentation is deliberately aggressive. §2.8 showed colour is a shortcut in this dataset;
randomising brightness, contrast, saturation and hue on every training step is the cheapest
available defence against the network latching onto it.


In [ ]:
def load_images(paths, size=CFG.IMG_SIZE):
    X = np.zeros((len(paths), size, size, 3), np.uint8)
    for i, p in enumerate(paths):
        with Image.open(p) as im:
            X[i] = np.asarray(im.convert("RGB").resize((size, size), Image.BILINEAR))
        if (i+1) % 500 == 0: print(f"   {i+1}/{len(paths)}")
    return X

t0 = time.time()
X_ALL = load_images(ds.path.tolist()); Y_ALL = ds.y.values.astype(np.int32)
print(f"loaded {X_ALL.shape} in {time.time()-t0:.0f}s  ({X_ALL.nbytes/1e6:.0f} MB)")

fig, axes = plt.subplots(2, K, figsize=(2*K, 4.4))
for c in range(K):
    idx = np.where(Y_ALL == c)[0][:2]
    for r in range(2):
        ax = axes[r, c]; ax.axis("off")
        if r < len(idx): ax.imshow(X_ALL[idx[r]])
        if r == 0: ax.set_title(f"{CLASS_NAMES[c]}\nn={int((Y_ALL==c).sum())}", fontsize=8)
plt.suptitle("Audited dataset — two examples per stage", y=.99); savefig("05_examples")


In [ ]:
from tensorflow.keras import layers, Model, Input

class ResNetPreprocess(layers.Layer):
    """Caffe-style ImageNet preprocessing, written out explicitly so the graph stays flat
    and Grad-CAM can reach into it (a Lambda wrapping preprocess_input breaks under Keras 3)."""
    MEAN = [103.939, 116.779, 123.68]
    def call(self, x):
        x = tf.cast(x, tf.float32)[..., ::-1]              # RGB -> BGR
        return x - tf.constant(self.MEAN, dtype=tf.float32)
    def compute_output_shape(self, s): return s

def make_augmenter():
    return tf.keras.Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.12, fill_mode="reflect"),
        layers.RandomZoom(0.15, 0.15, fill_mode="reflect"),
        layers.RandomTranslation(0.08, 0.08, fill_mode="reflect"),
        layers.RandomBrightness(0.25, value_range=(0, 255)),   # <- anti-shortcut
        layers.RandomContrast(0.25),
    ], name="augment")

def class_weights(y, K):
    cnt = np.bincount(y, minlength=K).astype(np.float64)
    w = len(y) / (K * np.maximum(cnt, 1)); w[cnt == 0] = 0
    return w

def sample_weights(y, K, use=CFG.USE_CLASS_WEIGHTS):
    return class_weights(y, K)[y].astype(np.float32) if use else np.ones(len(y), np.float32)

print("class weights (audited dataset):")
for c, w in enumerate(class_weights(Y_ALL, K)): print(f"   {CLASS_NAMES[c]:<12} n={int((Y_ALL==c).sum()):>4}  w={w:.2f}")


## 6 · Models

### 6.1 · Three heads on one backbone

All three share an identical ResNet-50 trunk so the comparison isolates the *head*, which is the
project's actual research question.

| head | outputs | training signal | what a wrong answer costs |
|---|---|---|---|
| `ce` | K softmax | categorical cross-entropy | the same, whatever the distance |
| `coral` | K−1 logits, **shared weights + per-threshold bias** | K−1 binary "is y > k?" tasks | grows with distance |
| `corn` | K−1 logits, independent weights | K−1 **conditional** binary tasks | grows with distance |

**CORAL** (Cao, Mirjalili & Raschka, *Pattern Recognition Letters* 2020) forces rank consistency
architecturally: all K−1 classifiers share one weight vector and differ only in bias, so
P(y>1) ≥ P(y>2) ≥ … is guaranteed. The cost is expressiveness — one direction in feature space
has to serve every threshold.

**CORN** (Shi, Cao & Raschka, *Pattern Analysis and Applications* 2023) gets the same guarantee
without the constraint. Head *j* estimates a **conditional** probability
P(y > r_j | y > r_{j−1}) and is trained only on the subset S_j = {i : y_i > r_{j−1}}. Unconditional
probabilities are rebuilt as a running product

  P̂(y > r_k) = ∏_{j≤k} f_j(x),

which is monotonically decreasing because every factor is in [0,1] — rank consistency falls out of
the chain rule. Predicted rank is q = 1 + Σ_j 1[P̂(y > r_j) > 0.5]. This is the
"ordinal cumulative-threshold formulation" named in the proposal, in its current best form.

A practical note on the loss: we return a **per-sample** value so Keras `sample_weight` (our class
balancing) composes correctly. Summing each sample's active tasks means a stage-7 example
contributes K−1 terms while a stage-1 example contributes one — which incidentally upweights the
rare advanced stages, on top of the explicit class weights.


In [ ]:
def build_model(head="corn", K=K, backbone="resnet50", dropout=0.35, trainable_backbone=False):
    """Flat functional graph -- no nested sub-model -- so model.get_layer(conv) works for Grad-CAM."""
    inp = Input((CFG.IMG_SIZE, CFG.IMG_SIZE, 3), name="image")
    x = make_augmenter()(inp)
    x = ResNetPreprocess(name="imagenet_preproc")(x)
    if backbone == "resnet50":
        base = tf.keras.applications.ResNet50(include_top=False, weights="imagenet", input_tensor=x)
        conv_name = "conv5_block3_out"
    elif backbone == "efficientnetb0":
        base = tf.keras.applications.EfficientNetB0(include_top=False, weights="imagenet", input_tensor=x)
        conv_name = "top_activation"
    else:
        raise ValueError(backbone)
    base.trainable = trainable_backbone
    f = layers.GlobalAveragePooling2D(name="gap")(base.output)
    f = layers.Dropout(dropout, name="drop")(f)

    if head == "ce":
        out = layers.Dense(K, activation="softmax", dtype="float32", name="ce_out")(f)
    elif head == "coral":
        shared = layers.Dense(1, use_bias=False, name="coral_shared")(f)          # (B,1)
        out = CoralBias(K - 1, name="coral_out")(shared)                          # (B,K-1)
    elif head == "corn":
        out = layers.Dense(K - 1, dtype="float32", name="corn_out")(f)            # (B,K-1) logits
    else:
        raise ValueError(head)
    m = Model(inp, out, name=f"{backbone}_{head}")
    m._conv_layer_name, m._head, m._K = conv_name, head, K
    return m

class CoralBias(layers.Layer):
    """CORAL: one shared projection + K-1 independent bias terms (Cao et al. 2020)."""
    def __init__(self, n_thresholds, **kw):
        super().__init__(**kw); self.n = n_thresholds
    def build(self, s):
        self.b = self.add_weight(shape=(self.n,), initializer="zeros",
                                 trainable=True, name="thresholds")   # kwargs: Keras 2 + 3 safe
    def call(self, x): return x + self.b                # broadcast (B,1)+(K-1,) -> (B,K-1)
    def compute_output_shape(self, s): return (s[0], self.n)
    def get_config(self): return {**super().get_config(), "n_thresholds": self.n}

print("heads available: ce | coral | corn")


In [ ]:
EPS = 1e-7

def corn_loss(K):
    """Shi, Cao & Raschka (2023). Task j trained only on the conditional subset {y >= j}."""
    def loss(y_true, logits):
        y = tf.cast(tf.reshape(y_true, [-1]), tf.int32)                     # (B,)
        j = tf.range(K - 1, dtype=tf.int32)[None, :]                        # (1,K-1)
        mask   = tf.cast(y[:, None] >= j, tf.float32)                       # in S_j ?
        target = tf.cast(y[:, None] >  j, tf.float32)                       # y > r_j ?
        bce = tf.nn.sigmoid_cross_entropy_with_logits(labels=target, logits=logits)
        return tf.reduce_sum(bce * mask, axis=1)                            # per-sample
    return loss

def coral_loss(K):
    """Cao et al. (2020): unconditional binary tasks 1[y > j] over all K-1 thresholds."""
    def loss(y_true, logits):
        y = tf.cast(tf.reshape(y_true, [-1]), tf.int32)
        j = tf.range(K - 1, dtype=tf.int32)[None, :]
        target = tf.cast(y[:, None] > j, tf.float32)
        return tf.reduce_sum(
            tf.nn.sigmoid_cross_entropy_with_logits(labels=target, logits=logits), axis=1)
    return loss

def ce_loss(K, smoothing=CFG.LABEL_SMOOTHING):
    cce = tf.keras.losses.CategoricalCrossentropy(label_smoothing=smoothing,
                                                  reduction="none")
    def loss(y_true, y_pred):
        y = tf.cast(tf.reshape(y_true, [-1]), tf.int32)
        return cce(tf.one_hot(y, K), y_pred)
    return loss

LOSSES = {"ce": ce_loss, "coral": coral_loss, "corn": corn_loss}

# ---------------------------------------------------------------------------------------
# Decoding: raw head output -> cumulative P(y>k), class posteriors, and a predicted stage.
# ---------------------------------------------------------------------------------------
def decode(out, head, K):
    """Returns (P_gt, proba, y_thresh, y_expected).
       P_gt      (N,K-1) : P(y > k)
       proba     (N,K)   : per-class posterior
       y_thresh  (N,)    : threshold rule  q = sum_k 1[P(y>k) > .5]      (paper rule)
       y_expected(N,)    : rounded expected value  round(sum_k P(y>k))   (QWK/MAE-friendly)"""
    out = np.asarray(out, np.float64)
    if head == "ce":
        proba = out / out.sum(1, keepdims=True)
        P_gt = np.cumsum(proba[:, ::-1], 1)[:, ::-1][:, 1:]     # P(y>k) = sum_{i>k} p_i
    else:
        s = 1. / (1. + np.exp(-out))
        P_gt = np.cumprod(s, 1) if head == "corn" else s        # CORN chains, CORAL is direct
        P_gt = np.minimum.accumulate(P_gt, 1)                   # enforce monotone (no-op for corn)
        pad = np.concatenate([np.ones((len(P_gt), 1)), P_gt, np.zeros((len(P_gt), 1))], 1)
        proba = np.clip(pad[:, :-1] - pad[:, 1:], EPS, 1)
        proba /= proba.sum(1, keepdims=True)
    return (P_gt,
            proba,
            (P_gt > .5).sum(1).astype(int),
            np.clip(np.round(P_gt.sum(1)), 0, K - 1).astype(int))

def rank_monotonicity_violations(out, head, K):
    """Measured on the RAW head output, BEFORE decode() enforces monotonicity.
       CE and CORN are rank-consistent by construction (so this is 0 by definition); CORAL only is
       if its learned biases happen to come out ordered, so for CORAL this is a real diagnostic."""
    out = np.asarray(out, np.float64)
    if head == "ce":
        pr = out / out.sum(1, keepdims=True); P = np.cumsum(pr[:, ::-1], 1)[:, ::-1][:, 1:]
    elif head == "corn":
        P = np.cumprod(1. / (1. + np.exp(-out)), 1)
    else:
        P = 1. / (1. + np.exp(-out))
    return float((np.diff(P, axis=1) > 1e-9).any(1).mean())

print("losses + decoders ready")


### 6.2 · Grad-CAM reachability check

The classic Grad-CAM failure on Colab is a *nested* backbone: wrapping `ResNet50` as a layer
inside a `Sequential` disconnects the graph and gradients come back `None`. `build_model` avoids
it by threading the augmentation and preprocessing tensors straight into `input_tensor=`, so every
ResNet layer is a first-class layer of the final model. We assert that here rather than discovering
it in §12 after an hour of training.


In [ ]:
_m = build_model("corn")
_names = {l.name for l in _m.layers}
assert _m._conv_layer_name in _names, "backbone is nested -- Grad-CAM would fail"
_g = Model(_m.inputs, [_m.get_layer(_m._conv_layer_name).output, _m.output])
_x = tf.convert_to_tensor(X_ALL[:2].astype("float32"))
with tf.GradientTape() as tape:
    _c, _o = _g(_x, training=False)          # tape records conv -> head, so _c is differentiable
    _s = tf.reduce_sum(tf.sigmoid(_o))
_grad = tape.gradient(_s, _c)
print("model layers              :", len(_m.layers))
print("conv feature map          :", _m._conv_layer_name, tuple(_c.shape))
print("gradient reaches it       :", _grad is not None)
assert _grad is not None, "Grad-CAM gradient is None -- the graph is disconnected"
print(f"params total / trainable  : {_m.count_params():,} / "
      f"{sum(int(np.prod(w.shape)) for w in _m.trainable_weights):,}")
print("\nGRAD-CAM REACHABILITY: OK")
del _m, _g, _c, _o, _grad; tf.keras.backend.clear_session()


## 7 · Metrics

Per the proposal, accuracy is not the headline. The four primary metrics are **macro-F1**,
**balanced accuracy**, **quadratic-weighted κ** and **MAE in stages**; everything else is
supporting. Two additions worth noting:

* **Adjacent accuracy (±1 stage).** Given that human raters achieve only ICC 0.63–0.68 on this
  scale, exact-match is arguably the wrong target and "within one stage" is the clinically
  honest one.
* **Expected calibration error.** A "confidence score" that is not calibrated is decoration;
  §11 fits a temperature on validation data and reports ECE before and after.


In [ ]:
from sklearn.metrics import (confusion_matrix, precision_recall_fscore_support,
                             roc_auc_score, classification_report)

def predicted_confidence(proba, y_pred, tol=0):
    """Posterior mass within `tol` stages of the reported prediction.

    tol=0 gives P(y = y_pred). Beware: the ordinal heads decode with the expected-value rule
    round(sum_k P(y>k)), which can land BETWEEN the modes of a bimodal posterior -- so this
    can be near zero even for a correct prediction. tol=1 gives P(|y - y_pred| <= 1), which is
    robust to that and is the clinically meaningful quantity given adjacent-stage ambiguity."""
    proba = np.asarray(proba); y = np.asarray(y_pred).astype(int)
    idx = np.arange(proba.shape[1])[None, :]
    return (proba * (np.abs(idx - y[:, None]) <= tol)).sum(1)

def expected_calibration_error(conf, correct, n_bins=15):
    b = np.linspace(0, 1, n_bins + 1); e = 0.
    for lo, hi in zip(b[:-1], b[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.sum(): e += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return float(e)

def ordinal_metrics(y_true, y_pred, proba=None, K=K, per_class=True):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    p, r, f1, sup = precision_recall_fscore_support(y_true, y_pred, labels=range(K), zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=range(K))
    spec = [( cm.sum() - cm[c].sum() - cm[:, c].sum() + cm[c, c] ) /
            max(1, cm.sum() - cm[c].sum()) for c in range(K)]
    m = dict(
        accuracy        = float((y_true == y_pred).mean()),
        adjacent_acc    = float((np.abs(y_true - y_pred) <= 1).mean()),
        macro_f1        = float(f1.mean()),
        balanced_acc    = float(balanced_accuracy_score(y_true, y_pred)),
        QWK             = float(cohen_kappa_score(y_true, y_pred, labels=range(K), weights="quadratic")),
        linear_kappa    = float(cohen_kappa_score(y_true, y_pred, labels=range(K), weights="linear")),
        MAE             = float(np.abs(y_true - y_pred).mean()),
        RMSE            = float(np.sqrt(((y_true - y_pred) ** 2).mean())),
        max_stage_error = int(np.abs(y_true - y_pred).max()) if len(y_true) else 0,
        macro_precision = float(p.mean()), macro_recall = float(r.mean()),
        mean_specificity= float(np.mean(spec)),
    )
    if proba is not None:
        proba = np.asarray(proba)
        present = [c for c in range(K) if (y_true == c).sum() > 0]
        try:
            m["ovr_auc_macro"] = float(roc_auc_score(
                np.eye(K)[y_true][:, present], proba[:, present], average="macro", multi_class="ovr"))
        except Exception: m["ovr_auc_macro"] = float("nan")
        # the confidence attached to a prediction is the posterior OF THAT prediction.
        # For the ordinal heads y_pred comes from round(sum_k P(y>k)), which is not always
        # the argmax, so max_c P(y=c) would score a different class than the one being checked.
        conf = predicted_confidence(proba, y_pred, 0)
        conf1 = predicted_confidence(proba, y_pred, 1)
        m["mean_confidence"] = float(conf.mean())
        m["mean_confidence_within1"] = float(conf1.mean())
        m["max_posterior_mean"] = float(proba.max(1).mean())
        m["ECE"] = expected_calibration_error(conf, (y_true == y_pred).astype(float))
        m["ECE_within1"] = expected_calibration_error(
            conf1, (np.abs(y_true - y_pred) <= 1).astype(float))
        # a large gap here means the posterior is bimodal and the point prediction sits in a
        # low-mass valley -- worth reporting, it is a real property of ordinal decoding
        m["posterior_valley_rate"] = float((proba.max(1) - conf > 0.25).mean())
    if per_class:
        m["_per_class"] = pd.DataFrame(dict(precision=p, recall=r, f1=f1, specificity=spec, support=sup),
                                       index=CLASS_NAMES[:K])
    return m

PRIMARY = ["QWK", "MAE", "macro_f1", "balanced_acc"]
SUPPORTING = ["accuracy", "adjacent_acc", "linear_kappa", "RMSE", "macro_precision",
              "macro_recall", "mean_specificity", "ovr_auc_macro", "ECE", "ECE_within1",
              "posterior_valley_rate"]

def bootstrap_ci(y_true, y_pred, proba=None, metrics=PRIMARY, n=CFG.N_BOOTSTRAP, alpha=.05, seed=CFG.SEED):
    rng = np.random.default_rng(seed); N = len(y_true); acc = {k: [] for k in metrics}
    for _ in range(n):
        b = rng.integers(0, N, N)
        if len(np.unique(y_true[b])) < 2: continue
        s = ordinal_metrics(y_true[b], y_pred[b], None if proba is None else proba[b], per_class=False)
        for k in metrics: acc[k].append(s[k])
    return {k: (float(np.percentile(v, 100*alpha/2)), float(np.percentile(v, 100*(1-alpha/2))))
            for k, v in acc.items() if v}

def plot_confusion(y_true, y_pred, title, K=K, fname=None):
    cm = confusion_matrix(y_true, y_pred, labels=range(K))
    cmn = cm / np.maximum(cm.sum(1, keepdims=True), 1)
    fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
    for a, M, t, fmt in [(ax[0], cm, "counts", "d"), (ax[1], cmn, "row-normalised (recall)", ".2f")]:
        im = a.imshow(M, cmap="YlGnBu", vmin=0, vmax=M.max() if fmt == "d" else 1)
        for i in range(K):
            for j in range(K):
                a.text(j, i, format(M[i, j], fmt), ha="center", va="center", fontsize=7,
                       color="white" if M[i, j] > (M.max() if fmt == "d" else 1) * .55 else "black")
        a.set(xticks=range(K), yticks=range(K), xlabel="predicted", ylabel="true", title=t)
        a.set_xticklabels([c.replace("Stage ", "") for c in CLASS_NAMES[:K]])
        a.set_yticklabels([c.replace("Stage ", "") for c in CLASS_NAMES[:K]])
        a.grid(False)
        for k in range(K):                    # mark the adjacent band
            for d in (-1, 1):
                if 0 <= k+d < K: a.add_patch(plt.Rectangle((k+d-.5, k-.5), 1, 1, fill=False, ec="#e76f51", lw=.8, ls=":"))
    off = np.abs(np.subtract.outer(range(K), range(K)))
    fig.suptitle(f"{title}\nexact {np.trace(cm)/cm.sum():.1%}  |  within +/-1 {(cm*(off<=1)).sum()/cm.sum():.1%}"
                 f"  |  >=3 stages off {(cm*(off>=3)).sum()/cm.sum():.1%}   (dotted = adjacent band)", y=1.04)
    savefig(fname or "confusion")
print("metric suite ready:", PRIMARY, "+", len(SUPPORTING), "supporting")


## 8 · Training

Two stages, as is standard for a small medical set: warm up the head on a frozen ImageNet trunk,
then fine-tune the upper blocks at a much lower learning rate with BatchNorm statistics frozen
(unfreezing BN on ~1k images destabilises training).

Model selection is on **val QWK**, not val accuracy — selecting on accuracy on a 38:1 imbalanced
ordinal problem picks the model that is best at saying "stage 1".


In [ ]:
class OrdinalValidation(tf.keras.callbacks.Callback):
    """Writes val_qwk / val_mae / val_macro_f1 into `logs` so EarlyStopping and
       ModelCheckpoint can monitor ordinal quality directly."""
    def __init__(self, Xv, yv, head, K, bs=64):
        super().__init__(); self.Xv, self.yv, self.head, self.K, self.bs = Xv, yv, head, K, bs
    def on_epoch_end(self, epoch, logs=None):
        logs = logs if logs is not None else {}
        out = self.model.predict(self.Xv, batch_size=self.bs, verbose=0)
        _, proba, y_thr, y_exp = decode(out, self.head, self.K)
        yp = y_exp if self.head != "ce" else np.argmax(proba, 1)
        logs["val_qwk"] = float(cohen_kappa_score(self.yv, yp, labels=range(self.K), weights="quadratic"))
        logs["val_mae"] = float(np.abs(self.yv - yp).mean())
        logs["val_macro_f1"] = float(f1_score(self.yv, yp, average="macro", labels=range(self.K), zero_division=0))

def set_finetune(model, from_layer=CFG.FINETUNE_FROM, freeze_bn=CFG.FREEZE_BN_ON_FINETUNE):
    names = [l.name for l in model.layers]
    # fall back to ~the top 40% of the stack when the named layer is not in this backbone
    start = names.index(from_layer) if from_layer in names else int(len(names) * .6)
    for i, l in enumerate(model.layers):
        l.trainable = i >= start
        if freeze_bn and isinstance(l, layers.BatchNormalization): l.trainable = False
    return sum(int(np.prod(w.shape)) for w in model.trainable_weights)

def train_model(head, tr, va, backbone="resnet50", X=None, Y=None, verbose=0, tag=""):
    X = X_ALL if X is None else X; Y = Y_ALL if Y is None else Y
    set_seed(CFG.SEED)
    Xtr, ytr, Xva, yva = X[tr], Y[tr], X[va], Y[va]
    sw = sample_weights(ytr, K)
    model = build_model(head, K, backbone, trainable_backbone=False)
    lossfn = LOSSES[head](K)
    ckpt = f"{CFG.OUT_DIR}/ckpt_{backbone}_{head}{tag}.weights.h5"

    cbs = lambda: [OrdinalValidation(Xva, yva, head, K),
                   tf.keras.callbacks.EarlyStopping(monitor=CFG.MONITOR, mode="max",
                        patience=CFG.EARLY_STOP_PATIENCE, restore_best_weights=True, verbose=verbose),
                   tf.keras.callbacks.ModelCheckpoint(ckpt, monitor=CFG.MONITOR, mode="max",
                        save_best_only=True, save_weights_only=True, verbose=0),
                   tf.keras.callbacks.ReduceLROnPlateau(monitor=CFG.MONITOR, mode="max",
                        factor=.3, patience=4, min_lr=1e-7, verbose=0)]

    model.compile(tf.keras.optimizers.Adam(CFG.LR_HEAD), loss=lossfn)
    h1 = model.fit(Xtr, ytr, sample_weight=sw, validation_data=(Xva, yva),
                   epochs=CFG.EPOCHS_HEAD, batch_size=CFG.BATCH_SIZE, callbacks=cbs(), verbose=verbose)

    ntrain = set_finetune(model)
    model.compile(tf.keras.optimizers.Adam(CFG.LR_FINETUNE), loss=lossfn)
    h2 = model.fit(Xtr, ytr, sample_weight=sw, validation_data=(Xva, yva),
                   epochs=CFG.EPOCHS_FINETUNE, batch_size=CFG.BATCH_SIZE, callbacks=cbs(), verbose=verbose)

    hist = {k: list(h1.history.get(k, [])) + list(h2.history.get(k, [])) for k in
            set(h1.history) | set(h2.history)}
    hist["_stage_boundary"] = len(h1.history.get("loss", []))
    return model, hist

def predict(model, X, head=None, K=K, bs=64):
    head = head or model._head
    out = model.predict(X, batch_size=bs, verbose=0)
    P, proba, y_thr, y_exp = decode(out, head, K)
    yp = np.argmax(proba, 1) if head == "ce" else y_exp
    return dict(raw=out, P_gt=P, proba=proba, y_pred=yp, y_thresh=y_thr, y_expected=y_exp)

def plot_history(hist, title):
    fig, ax = plt.subplots(1, 3, figsize=(12, 3))
    for a, keys, lab in [(ax[0], ["loss", "val_loss"], "loss"),
                         (ax[1], ["val_qwk"], "val QWK"), (ax[2], ["val_mae"], "val MAE (stages)")]:
        for k in keys:
            if k in hist: a.plot(hist[k], label=k)
        if "_stage_boundary" in hist: a.axvline(hist["_stage_boundary"] - .5, ls="--", c="#bc4749", lw=1)
        a.set(xlabel="epoch", ylabel=lab); a.legend(fontsize=7)
    fig.suptitle(f"{title}   (dashed = head-only -> fine-tune)", y=1.03)
    savefig("08_history_" + title.replace(" ", "_").replace("/", "_"))
print("training utilities ready")


## 9 · Baselines

The proposal calls for ResNet-50 to be compared against "lighter CNN and classical SVM baselines".
We add two more that matter more than either:

* **colour-only logistic regression** — the shortcut floor from §2.11. Any model that fails to
  beat it has learnt nothing about scalps.
* **majority class** — the floor below the floor.

The classical baseline is an SVM on frozen ImageNet ResNet-50 features (a far stronger classical
baseline than HOG/SIFT, so the deep models have to actually earn their win), in both a plain
multi-class and an ordinal cumulative-threshold form.


In [ ]:
from sklearn.svm import SVC, LinearSVC
from sklearn.calibration import CalibratedClassifierCV

_feat_extractor = None
def imagenet_features(X, bs=64):
    """Frozen ResNet-50 GAP features (2048-d) for the classical baselines."""
    global _feat_extractor
    if _feat_extractor is None:
        inp = Input((CFG.IMG_SIZE, CFG.IMG_SIZE, 3))
        p = ResNetPreprocess()(inp)
        b = tf.keras.applications.ResNet50(include_top=False, weights="imagenet", input_tensor=p)
        _feat_extractor = Model(inp, layers.GlobalAveragePooling2D()(b.output))
    return _feat_extractor.predict(X, batch_size=bs, verbose=0)

def baseline_majority(tr, va, **kw):
    c = np.bincount(Y_ALL[tr], minlength=K).argmax()
    p = np.bincount(Y_ALL[tr], minlength=K) / len(tr)
    return np.full(len(va), c), np.tile(p, (len(va), 1))

def baseline_colour(tr, va, **kw):
    """The shortcut floor from 2.11, re-fitted per fold on the audited data."""
    m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, class_weight="balanced"))
    m.fit(COLOUR_X_DS[tr], Y_ALL[tr])
    return m.predict(COLOUR_X_DS[va]), m.predict_proba(COLOUR_X_DS[va])

def baseline_svm(tr, va, ordinal=False, **kw):
    Ftr, Fva = FEAT[tr], FEAT[va]
    sc = StandardScaler().fit(Ftr); Ftr, Fva = sc.transform(Ftr), sc.transform(Fva)
    if not ordinal:
        m = SVC(C=1.0, kernel="rbf", gamma="scale", class_weight="balanced", probability=True,
                random_state=CFG.SEED).fit(Ftr, Y_ALL[tr])
        return m.predict(Fva), m.predict_proba(Fva)
    # ordinal SVM: K-1 cumulative binary problems, decoded exactly like CORAL
    P = np.zeros((len(va), K - 1))
    for j in range(K - 1):
        yb = (Y_ALL[tr] > j).astype(int)
        if len(np.unique(yb)) < 2:
            P[:, j] = float(yb[0]); continue
        m = CalibratedClassifierCV(LinearSVC(C=.1, class_weight="balanced", max_iter=5000,
                                             random_state=CFG.SEED), cv=3).fit(Ftr, yb)
        P[:, j] = m.predict_proba(Fva)[:, 1]
    P = np.minimum.accumulate(P, 1)
    pad = np.concatenate([np.ones((len(P), 1)), P, np.zeros((len(P), 1))], 1)
    proba = np.clip(pad[:, :-1] - pad[:, 1:], EPS, 1); proba /= proba.sum(1, keepdims=True)
    return np.clip(np.round(P.sum(1)), 0, K - 1).astype(int), proba

def baseline_small_cnn(tr, va, **kw):
    """The 'lighter CNN' the proposal asks for: trained from scratch, ~0.4M params."""
    set_seed(CFG.SEED)
    inp = Input((CFG.IMG_SIZE, CFG.IMG_SIZE, 3)); x = make_augmenter()(inp)
    x = layers.Rescaling(1 / 127.5, offset=-1)(x)
    for f in (32, 64, 128, 128):
        x = layers.Conv2D(f, 3, padding="same", use_bias=False)(x)
        x = layers.BatchNormalization()(x); x = layers.ReLU()(x); x = layers.MaxPooling2D()(x)
    x = layers.GlobalAveragePooling2D()(x); x = layers.Dropout(.3)(x)
    out = layers.Dense(K, activation="softmax", dtype="float32")(x)
    m = Model(inp, out); m.compile(tf.keras.optimizers.Adam(1e-3), loss=ce_loss(K))
    m.fit(X_ALL[tr], Y_ALL[tr], sample_weight=sample_weights(Y_ALL[tr], K),
          validation_data=(X_ALL[va], Y_ALL[va]),
          epochs=CFG.EPOCHS_HEAD + CFG.EPOCHS_FINETUNE, batch_size=CFG.BATCH_SIZE, verbose=0,
          callbacks=[OrdinalValidation(X_ALL[va], Y_ALL[va], "ce", K),
                     tf.keras.callbacks.EarlyStopping(monitor="val_qwk", mode="max",
                        patience=CFG.EARLY_STOP_PATIENCE, restore_best_weights=True)])
    p = m.predict(X_ALL[va], batch_size=64, verbose=0)
    del m; tf.keras.backend.clear_session()
    return np.argmax(p, 1), p

# Colour features realigned to the canonical `ds` row order via the retained man_idx.
COLOUR_X_DS = COLOUR_X[ds.man_idx.values]
assert len(COLOUR_X_DS) == len(ds)

print("extracting frozen ImageNet features for the classical baselines ...")
FEAT = imagenet_features(X_ALL); print("features:", FEAT.shape)

BASELINES = {
    "majority-class":       baseline_majority,
    "colour-only LR":       baseline_colour,
    "SVM (frozen RN50)":    lambda tr, va, **k: baseline_svm(tr, va, ordinal=False),
    "ordinal SVM (frozen)": lambda tr, va, **k: baseline_svm(tr, va, ordinal=True),
    "small CNN (scratch)":  baseline_small_cnn,
}
print("baselines:", list(BASELINES))


## 10 · Benchmark — cross-validated on the development set

Primary comparison. Every model sees identical folds. Metrics are pooled over the out-of-fold
predictions, which is what makes the rare stages measurable at all: stage 6 gets ~15 test
predictions across the sweep instead of ~4 from a single hold-out.

**Runtime.** ResNet-50 on ~925 images is roughly 8–12 min per fold per model on a T4.
`CFG.BENCH_FOLDS = 3` with four deep models ≈ 2 hours. Lower `BENCH_FOLDS` to 2, or drop a head,
if your session is short. `QUICK_RUN` does one fold with tiny epoch counts purely to prove the
code runs end to end.


In [ ]:
CFG.BENCH_FOLDS = 1 if CFG.QUICK_RUN else 3
DEEP_MODELS = {
    "ResNet50 + CE (softmax)":  dict(head="ce",    backbone="resnet50"),
    "ResNet50 + CORAL":         dict(head="coral", backbone="resnet50"),
    "ResNet50 + CORN":          dict(head="corn",  backbone="resnet50"),
    "EfficientNetB0 + CORN":    dict(head="corn",  backbone="efficientnetb0"),
}
bench_folds = folds[:CFG.BENCH_FOLDS]
print(f"benchmarking {len(BASELINES)} baselines + {len(DEEP_MODELS)} deep models "
      f"over {len(bench_folds)} fold(s)\n")

OOF, RESULTS, HISTORIES = {}, [], {}

for name, fn in BASELINES.items():
    yt, yp, pr = [], [], []
    for f, (tr, va) in enumerate(bench_folds):
        a, b = fn(tr, va); yt.append(Y_ALL[va]); yp.append(a); pr.append(b)
    yt, yp, pr = np.concatenate(yt), np.concatenate(yp), np.concatenate(pr)
    OOF[name] = (yt, yp, pr)
    m = ordinal_metrics(yt, yp, pr); RESULTS.append(dict(model=name, family="baseline", **{k: m[k] for k in PRIMARY + SUPPORTING}))
    print(f"  {name:<24} QWK {m['QWK']:.3f}  MAE {m['MAE']:.2f}  macroF1 {m['macro_f1']:.3f}  balAcc {m['balanced_acc']:.3f}")

print()
for name, spec in DEEP_MODELS.items():
    yt, yp, pr, raws = [], [], [], []
    for f, (tr, va) in enumerate(bench_folds):
        t0 = time.time()
        model, hist = train_model(spec["head"], tr, va, spec["backbone"], tag=f"_f{f}")
        p = predict(model, X_ALL[va], spec["head"])
        yt.append(Y_ALL[va]); yp.append(p["y_pred"]); pr.append(p["proba"]); raws.append(p["raw"])
        if f == 0: HISTORIES[name] = hist
        print(f"  {name:<24} fold {f}  QWK {cohen_kappa_score(Y_ALL[va], p['y_pred'], labels=range(K), weights='quadratic'):.3f}"
              f"  ({time.time()-t0:.0f}s)")
        del model; tf.keras.backend.clear_session()
    yt, yp, pr = np.concatenate(yt), np.concatenate(yp), np.concatenate(pr)
    OOF[name] = (yt, yp, pr)
    m = ordinal_metrics(yt, yp, pr)
    viol = float(np.mean([rank_monotonicity_violations(r, spec["head"], K) for r in raws]))
    RESULTS.append(dict(model=name, family="deep", rank_violations=viol, **{k: m[k] for k in PRIMARY + SUPPORTING}))
    print(f"  {name:<24} POOLED QWK {m['QWK']:.3f}  MAE {m['MAE']:.2f}  macroF1 {m['macro_f1']:.3f}  "
          f"rank-inconsistent preds {viol:.1%}\n")

RES = pd.DataFrame(RESULTS).set_index("model")
RES.to_csv(f"{CFG.OUT_DIR}/benchmark_cv.csv")


In [ ]:
show = RES[["family"] + PRIMARY + ["accuracy", "adjacent_acc", "ovr_auc_macro", "ECE"]].copy()
show = show.sort_values("QWK", ascending=False).round(3)
print("CROSS-VALIDATED BENCHMARK (pooled out-of-fold, development set)\n")
display(show)

best_name = show.index[0]
_deep_rows = show[show.family == "deep"]
best_deep_name = _deep_rows.index[0] if len(_deep_rows) else best_name
print(f"\nbest overall by QWK : {best_name}")
print(f"best DEEP model     : {best_deep_name}   <- this is what 11 trains and locks in")
if best_name != best_deep_name:
    print(f"""
  !!  A BASELINE ({best_name}) OUTRANKS EVERY DEEP MODEL.
      Do not bury this. On ~1.4k independent images with 19 examples in the rarest stage,
      a frozen-feature SVM beating a fine-tuned ResNet-50 is an entirely plausible honest
      outcome -- and it says more about the dataset's size than any head comparison does.
      Report it, and check the fold-to-fold spread before calling the gap real.""")
print(f"colour-only shortcut floor on the VENDOR split was QWK {COLOUR_VENDOR['QWK']:.3f} -- for context,")
print("that is what a model with no access to scalp structure scored on the unaudited data.")
print(f"human inter-observer reliability on this scale: ICC 0.63-0.68 (Guarrera et al. 2009).")

fig, ax = plt.subplots(1, 3, figsize=(13, 3.6))
o = show.sort_values("QWK")
cols = ["#2a9d8f" if f == "deep" else "#adb5bd" for f in o.family]
ax[0].barh(range(len(o)), o.QWK, color=cols); ax[0].set_yticks(range(len(o)), o.index, fontsize=7)
ax[0].axvline(COLOUR_AUDITED["QWK"], ls=":", c="#bc4749"); ax[0].set(xlabel="quadratic-weighted kappa")
ax[0].text(COLOUR_AUDITED["QWK"], -.6, " colour-only", color="#bc4749", fontsize=7)
o2 = show.sort_values("MAE", ascending=False)
ax[1].barh(range(len(o2)), o2.MAE, color=["#2a9d8f" if f == "deep" else "#adb5bd" for f in o2.family])
ax[1].set_yticks(range(len(o2)), [""]*len(o2)); ax[1].set(xlabel="MAE (stages)")
ax[2].scatter(show.macro_f1, show.QWK, c=["#2a9d8f" if f == "deep" else "#adb5bd" for f in show.family], s=60)
for n, r in show.iterrows(): ax[2].annotate(n.split(" +")[0][:16], (r.macro_f1, r.QWK), fontsize=6, xytext=(3,3), textcoords="offset points")
ax[2].set(xlabel="macro-F1", ylabel="QWK")
fig.suptitle("Benchmark on the audited protocol (green = deep, grey = baseline)", y=1.04)
savefig("10_benchmark")

for n, h in HISTORIES.items(): plot_history(h, n)

fig, axes = plt.subplots(1, min(3, len(DEEP_MODELS)), figsize=(4.2*min(3,len(DEEP_MODELS)), 3.6))
for ax_, n in zip(np.atleast_1d(axes), list(DEEP_MODELS)[:3]):
    yt, yp, _ = OOF[n]
    cm = confusion_matrix(yt, yp, labels=range(K)); cmn = cm/np.maximum(cm.sum(1,keepdims=True),1)
    ax_.imshow(cmn, cmap="YlGnBu", vmin=0, vmax=1); ax_.grid(False)
    for i in range(K):
        for j in range(K): ax_.text(j, i, f"{cmn[i,j]:.2f}", ha="center", va="center", fontsize=6,
                                    color="white" if cmn[i,j] > .55 else "black")
    ax_.set(title=n, xlabel="predicted", ylabel="true", xticks=range(K), yticks=range(K))
fig.suptitle("Out-of-fold confusion (row-normalised) — mass should hug the diagonal band", y=1.05)
savefig("10_confusions")


### 10.1 · Does ordinal modelling actually help?

This is the project's central research question, so it deserves a direct answer rather than a
table the reader has to interpret. We compare the softmax head against the ordinal heads on the
*same folds*, on the metrics that encode order.


In [ ]:
ce_key = "ResNet50 + CE (softmax)"
comp = []
for n in DEEP_MODELS:
    yt, yp, pr = OOF[n]; m = ordinal_metrics(yt, yp, pr)
    off = np.abs(yt - yp)
    comp.append(dict(model=n, QWK=m["QWK"], MAE=m["MAE"], macro_f1=m["macro_f1"],
                     exact=m["accuracy"], within_1=m["adjacent_acc"],
                     pct_err_ge2=float((off >= 2).mean()), pct_err_ge3=float((off >= 3).mean()),
                     worst=int(off.max())))
comp = pd.DataFrame(comp).set_index("model").round(3); display(comp)

if ce_key in comp.index:
    base = comp.loc[ce_key]
    d = (comp - base).drop(index=ce_key).round(3)
    print(f"\nDelta vs. the softmax baseline (positive QWK / negative MAE & err>=2 = ordinal head wins):")
    display(d[["QWK", "MAE", "macro_f1", "pct_err_ge2", "pct_err_ge3"]])

fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
w = .8/len(comp)
for i, (n, r) in enumerate(comp.iterrows()):
    off = np.abs(OOF[n][0] - OOF[n][1]); h = np.bincount(off, minlength=K)/len(off)
    ax[0].bar(np.arange(K)+i*w-.4, h, w, label=n.split(" +")[0][:18])
ax[0].set(xlabel="|true - predicted| (stages)", ylabel="fraction", title="Stage-distance error profile")
ax[0].legend(fontsize=6)
ax[1].bar(range(len(comp)), comp.pct_err_ge2, color="#bc4749")
ax[1].set_xticks(range(len(comp)), [n.split(" +")[0][:14] for n in comp.index], rotation=20, fontsize=7)
ax[1].set(ylabel="fraction of predictions >=2 stages off", title="Large, clinically consequential errors")
savefig("10_ordinal_vs_ce")

print("""
How to read this honestly:
  * QWK and MAE are the metrics that encode order. If CORAL/CORN beat CE there while tying on
    macro-F1, the ordinal formulation is doing exactly what it claims: same amount right, but the
    mistakes it does make land closer to the truth.
  * If the ordinal heads DO NOT win, report that. A negative answer to the research question is a
    result -- and with ~1.4k images and 19 stage-6 examples, fold-to-fold noise can easily exceed
    the effect size. Check the CIs in 11 before claiming any difference is real.""")


## 11 · Locked-test evaluation — run this once

The test set has not been read by anything above. We now train the winning configuration on the
full development set (with a small internal validation carve-out for early stopping and for
fitting the calibration temperature), then evaluate **once**.

Confidence scores are temperature-scaled on the internal validation set — an uncalibrated softmax
maximum is not a confidence, and the proposal promises calibrated ones.


In [ ]:
from scipy.optimize import minimize_scalar

FINAL_NAME = best_deep_name              # always a deep model: 11 fine-tunes and calibrates it
BEST = DEEP_MODELS[FINAL_NAME]
print(f"final configuration: {FINAL_NAME}  ->  {BEST}")
if best_name != FINAL_NAME:
    print(f"(the overall benchmark winner was the baseline '{best_name}'; it needs no fine-tuning\n"
          f" stage, so 11 locks in the best deep model and you report both.)")

inner = StratifiedKFold(6, shuffle=True, random_state=CFG.SEED)
fit_idx, val_idx = next(iter(inner.split(np.zeros(len(dev_idx)), Y_ALL[dev_idx])))
fit_idx, val_idx = dev_idx[fit_idx], dev_idx[val_idx]
print(f"train {len(fit_idx)} | internal val {len(val_idx)} | LOCKED TEST {len(test_idx)}")

set_seed(CFG.SEED)
final_model, final_hist = train_model(BEST["head"], fit_idx, val_idx, BEST["backbone"], verbose=1, tag="_final")
plot_history(final_hist, "final model")
final_model.save_weights(f"{CFG.OUT_DIR}/final_model.weights.h5")

def apply_temperature(raw, T, head):
    return np.asarray(raw) / T if head != "ce" else np.exp(np.log(np.clip(raw, EPS, 1)) / T)

def fit_temperature(raw_val, y_val, head, K):
    def nll(logT):
        _, proba, _, _ = decode(apply_temperature(raw_val, np.exp(logT), head), head, K)
        proba = proba / proba.sum(1, keepdims=True)
        return -np.mean(np.log(np.clip(proba[np.arange(len(y_val)), y_val], EPS, 1)))
    r = minimize_scalar(nll, bounds=(np.log(.2), np.log(8.)), method="bounded")
    return float(np.exp(r.x))

pv = predict(final_model, X_ALL[val_idx], BEST["head"])
T = fit_temperature(pv["raw"], Y_ALL[val_idx], BEST["head"], K)
print(f"\ncalibration temperature fitted on internal val: T = {T:.3f}  ({'over' if T>1 else 'under'}-confident before scaling)")

# ---------------- the single evaluation ----------------
pt = predict(final_model, X_ALL[test_idx], BEST["head"])
_, proba_cal, _, _ = decode(apply_temperature(pt["raw"], T, BEST["head"]), BEST["head"], K)
y_true_t, y_pred_t = Y_ALL[test_idx], pt["y_pred"]

M_raw = ordinal_metrics(y_true_t, y_pred_t, pt["proba"])
M_cal = ordinal_metrics(y_true_t, y_pred_t, proba_cal)
CI = bootstrap_ci(y_true_t, y_pred_t, proba_cal, metrics=PRIMARY + ["accuracy", "adjacent_acc"])

print("\n" + "="*78); print(f"  LOCKED TEST SET  (n={len(test_idx)})   split lock {LOCK[:16]}..."); print("="*78)
for k in PRIMARY + ["accuracy", "adjacent_acc"]:
    lo, hi = CI.get(k, (float("nan"),)*2)
    print(f"  {k:<16} {M_cal[k]:>7.3f}   95% CI [{lo:.3f}, {hi:.3f}]")
print(f"  {'ECE (uncal.)':<16} {M_raw['ECE']:>7.3f}")
print(f"  {'ECE (T-scaled)':<16} {M_cal['ECE']:>7.3f}")
print(f"  {'ECE within +/-1':<16} {M_cal['ECE_within1']:>7.3f}   (conf = P(|y-y_pred|<=1))")
print(f"  {'valley rate':<16} {M_cal['posterior_valley_rate']:>7.1%}   "
      f"fraction where the point prediction sits in a low-mass valley")
print(f"  {'OvR AUC macro':<16} {M_cal['ovr_auc_macro']:>7.3f}")
print(f"  {'worst error':<16} {M_cal['max_stage_error']:>7d} stages")
print("="*78)
print("\nPER-CLASS (note the support column before reading any of these as reliable)")
display(M_cal["_per_class"].round(3))
plot_confusion(y_true_t, y_pred_t, f"Locked test set — {FINAL_NAME}", fname="11_confusion_locked")

# ---------------------------------------------------------------------------------------
# RESIDUAL-SIMILARITY SENSITIVITY CHECK
# Grouping (2.6) removes copies of the same PHOTOGRAPH. It cannot remove a second photo of
# the same PATIENT taken in another session -- there are no patient IDs in this dataset. If
# such pairs straddle dev/test they inflate these numbers. We cannot fix that, but we can
# bound it: re-score on the subset of test images that have no close dev neighbour.
# ---------------------------------------------------------------------------------------
_teb = bits[ds.loc[test_idx, "man_idx"].values].astype(np.int16)
_dvb = bits[ds.loc[dev_idx, "man_idx"].values].astype(np.int16)
nn_dist = np.empty(len(test_idx), int)
for _i in range(0, len(_teb), 256):
    nn_dist[_i:_i+256] = (_teb[_i:_i+256, None, :] != _dvb[None, :, :]).sum(2).min(1)
_rng = np.random.default_rng(CFG.SEED)
_a, _b = _rng.integers(0, len(bits), 20000), _rng.integers(0, len(bits), 20000)
_null = (bits[_a] != bits[_b]).sum(1)
print(f"\nRESIDUAL SIMILARITY  (unrelated image pairs differ by {_null.mean():.0f}+/-{_null.std():.0f} bits)")
rows = [dict(kept="all test images", n=len(test_idx),
             **{k: M_cal[k] for k in ["QWK", "MAE", "macro_f1", "accuracy"]})]
for thr in [6, 8, 10, 12]:
    keep = nn_dist > thr
    if keep.sum() >= 40 and len(np.unique(y_true_t[keep])) >= 3:
        mm = ordinal_metrics(y_true_t[keep], y_pred_t[keep], proba_cal[keep])
        rows.append(dict(kept=f"no dev neighbour within {thr} bits", n=int(keep.sum()),
                         **{k: mm[k] for k in ["QWK", "MAE", "macro_f1", "accuracy"]}))
SENS = pd.DataFrame(rows).set_index("kept").round(3); display(SENS)
SENS.to_csv(f"{CFG.OUT_DIR}/residual_similarity_sensitivity.csv")
print("""If the metrics hold up as the threshold rises, residual patient-level leakage is not
driving the result. If they fall away, say so plainly -- it is a limit of the DATASET
(no patient IDs), not of your method, and quantifying it is itself a contribution.""")

# ---- per-image predictions, so any later subset analysis needs no retraining ----
PRED = pd.DataFrame(dict(
    filename=ds.loc[test_idx, "filename"].values, group=ds.loc[test_idx, "group"].values,
    family=ds.loc[test_idx, "family"].values, true_stage=y_true_t + 1, pred_stage=y_pred_t + 1,
    abs_error=np.abs(y_true_t - y_pred_t), nn_dev_hamming=nn_dist,
    confidence=predicted_confidence(proba_cal, y_pred_t, 0),
    confidence_within1=predicted_confidence(proba_cal, y_pred_t, 1),
    max_posterior=proba_cal.max(1)))
for _c in range(K): PRED[f"p_stage{_c+1}"] = proba_cal[:, _c]
PRED.to_csv(f"{CFG.OUT_DIR}/test_predictions.csv", index=False)
print(f"\nper-image test predictions -> {CFG.OUT_DIR}/test_predictions.csv")


In [ ]:
# calibration + per-class reliability
fig, ax = plt.subplots(1, 3, figsize=(13, 3.4))
for lab, pr, c, tol in [("exact, uncalibrated", pt["proba"], "#bc4749", 0),
                        (f"exact, T={T:.2f}", proba_cal, "#2a9d8f", 0),
                        (f"within +/-1, T={T:.2f}", proba_cal, "#264653", 1)]:
    conf = predicted_confidence(pr, y_pred_t, tol)
    corr = (np.abs(y_pred_t - y_true_t) <= tol).astype(float)
    b = np.linspace(0, 1, 11); xs, ys = [], []
    for lo, hi in zip(b[:-1], b[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.sum() >= 3: xs.append(conf[m].mean()); ys.append(corr[m].mean())
    ax[0].plot(xs, ys, "o-", color=c, label=lab)
ax[0].plot([0,1],[0,1],"k:",lw=1); ax[0].set(xlabel="confidence", ylabel="empirical accuracy", title="Reliability diagram")
ax[0].legend(fontsize=7)

err = np.abs(y_true_t - y_pred_t)
ax[1].hist(err, bins=np.arange(-.5, K+.5), color="#457b9d", rwidth=.8)
ax[1].set(xlabel="|true - predicted| (stages)", ylabel="count", title="Stage-distance error distribution")
for d in range(int(err.max())+1): ax[1].text(d, (err==d).sum()+.5, f"{(err==d).mean():.0%}", ha="center", fontsize=7)

pc = M_cal["_per_class"]
ax[2].barh(range(K), pc.recall, color=STAGE_COLORS[:K])
for i, (r, s) in enumerate(zip(pc.recall, pc.support)):
    ci = 1.96*np.sqrt(max(r*(1-r),.01)/max(s,1))
    ax[2].errorbar(r, i, xerr=ci, color="k", capsize=2, lw=.8)
    ax[2].text(1.02, i, f"n={int(s)}", fontsize=6, va="center")
ax[2].set(yticks=range(K), yticklabels=[c.replace("Stage ","S") for c in CLASS_NAMES[:K]],
          xlabel="recall", xlim=(0,1.15), title="Per-class recall (Wald 95% CI)")
savefig("11_calibration_errors")

print(f"""
Read the right-hand panel before quoting any per-class number. With n={int(pc.support.min())} test
images in the smallest class, its recall CI spans most of the [0,1] interval -- that class is
effectively unmeasured on a single hold-out. The pooled cross-validated figures in 10 are the
defensible per-class evidence; this locked-test run is a confirmatory check that nothing in the
development loop overfit the folds.""")


## 12 · Explainability — Grad-CAM

> *"Grad-CAM heatmaps reviewed for attention to clinically relevant regions (hairline, vertex,
> scalp exposure); focus on irrelevant background is flagged as shortcut learning."* — proposal §6

For the softmax head the target is the predicted class score, as usual. For the ordinal heads we
differentiate the **predicted-class posterior** P(y = ŷ), reconstructed from the cumulative outputs.
The tempting alternative, the expected stage Σ_k P(y > k), is degenerate at low stages: for a
confident stage-1 prediction every P(y > k) ≈ 0, so its gradient is ~0 and the map is close to
noise, which is easily misread as "the model looks at the background". `gradcam(..., target="severity")`
is kept for advanced-stage inspection only.


In [ ]:
import cv2

_GRADCAM_CACHE = {}
def _grad_model(model):
    """Build the (conv_features, head_output) model once per model -- 12.1 calls gradcam()
       once per test image and rebuilding a Keras Model each time is minutes of overhead."""
    key = id(model)
    if key not in _GRADCAM_CACHE:
        conv = model.get_layer(model._conv_layer_name)
        _GRADCAM_CACHE[key] = Model(model.inputs, [conv.output, model.output])
    return _GRADCAM_CACHE[key]

def gradcam(model, x, head, K, target="class", class_idx=None):
    """x: (H,W,3) uint8/float. Returns a (H,W) heatmap in [0,1].

    target="class"    : gradient of the predicted-class posterior P(y=c).  DEFAULT.
    target="severity" : gradient of the expected stage sum_k P(y>k).
                        Intuitive, but DEGENERATE AT LOW STAGES -- for a confident stage-1
                        prediction every P(y>k) is ~0, so the gradient is ~0 and the map is
                        essentially noise. That produces a stage-dependent diffuseness which
                        is easily mistaken for the model "looking at the background". Use it
                        only for advanced stages, and never for a cross-stage comparison."""
    gm = _grad_model(model)
    xt = tf.convert_to_tensor(x[None].astype("float32"))
    with tf.GradientTape() as tape:
        fmap, out = gm(xt, training=False)   # the tape records conv -> head in this one pass
        if head == "ce":
            j = class_idx if class_idx is not None else tf.argmax(out[0])
            score = out[0, j]
        else:
            s = tf.sigmoid(out)
            P = tf.math.cumprod(s, axis=1) if head == "corn" else s
            if target == "severity":
                score = tf.reduce_sum(P)
            else:
                pad = tf.concat([tf.ones((1, 1)), P, tf.zeros((1, 1))], axis=1)
                proba = pad[:, :-1] - pad[:, 1:]              # P(y=c), differentiable
                j = class_idx if class_idx is not None else tf.argmax(proba[0])
                score = proba[0, j]
    g = tape.gradient(score, fmap)
    if g is None: raise RuntimeError("no gradient reached the conv map")
    w = tf.reduce_mean(g, axis=(0, 1, 2))
    cam = tf.nn.relu(tf.reduce_sum(fmap[0] * w, axis=-1)).numpy()
    cam = cv2.resize(cam, (x.shape[1], x.shape[0]))
    return cam / (cam.max() + 1e-8)

def overlay(img, cam, alpha=.45):
    hm = cv2.applyColorMap(np.uint8(255*cam), cv2.COLORMAP_JET)[:, :, ::-1]
    return np.uint8(np.clip((1-alpha)*img + alpha*hm, 0, 255))

_t0 = time.time(); _ = gradcam(final_model, X_ALL[test_idx[0]], BEST["head"], K)
print(f"Grad-CAM operational on the final model (first call {time.time()-_t0:.1f}s, "
      f"subsequent calls reuse the cached graph).")


In [ ]:
# one correct + one incorrect example per stage
order = np.argsort(np.abs(y_true_t - y_pred_t))
picks = []
for c in range(K):
    idx = [i for i in order if y_true_t[i] == c]
    if not idx: continue
    picks.append(("correct" if y_pred_t[idx[0]] == c else f"off by {abs(y_pred_t[idx[0]]-c)}", c, idx[0]))
    wrong = [i for i in idx[::-1] if y_pred_t[i] != c]
    if wrong: picks.append((f"off by {abs(y_pred_t[wrong[0]]-c)}", c, wrong[0]))

n = len(picks); cols = min(7, n); rows = int(np.ceil(n/cols))
fig, axes = plt.subplots(2*rows, cols, figsize=(2.1*cols, 4.4*rows), squeeze=False)
for k, (tagtxt, c, i) in enumerate(picks):
    r, cc = divmod(k, cols); img = X_ALL[test_idx[i]]
    cam = gradcam(final_model, img, BEST["head"], K)
    conf = proba_cal[i].max()
    for rr, pic, ttl in [(2*r, img, f"true S{c+1} / pred S{y_pred_t[i]+1}\n{tagtxt}, conf {conf:.2f}"),
                         (2*r+1, overlay(img, cam), "Grad-CAM")]:
        axes[rr, cc].imshow(pic); axes[rr, cc].axis("off"); axes[rr, cc].set_title(ttl, fontsize=6.5)
for k in range(n, rows*cols):
    r, cc = divmod(k, cols); axes[2*r, cc].axis("off"); axes[2*r+1, cc].axis("off")
plt.suptitle("Grad-CAM on the locked test set — gradient of the predicted-class posterior", y=.995)
savefig("12_gradcam_panel")


### 12.1 · Quantifying shortcut attention

Eyeballing heatmaps is not evidence. Given §2.9 (source collection correlates with severity), the
specific risk is the model reading *background* — a clinic wall, a shirt collar, a lighting rig.

**Read the by-stage panel with care.** Grad-CAM diffuseness is not constant across stages. If you
switch the target to `"severity"`, a confident stage-1 prediction has every P(y>k) ≈ 0, so the
gradient is near-zero and the map degenerates to noise — which looks exactly like "the model is
staring at the background". The default target here is the predicted-class posterior P(y=c),
which is well-scaled at every stage; a residual monotone trend under *that* target is evidence
worth reporting, one under the severity target is not.

These photographs are roughly head-centred, so we use a geometric proxy: the fraction of Grad-CAM
mass falling outside a centred ellipse covering ~55% of the frame. High peripheral mass on a
confident prediction is a red flag. This is a proxy, not scalp segmentation — report it as such,
and pair it with the manual review grid above.


In [ ]:
H = W = CFG.IMG_SIZE
yy, xx = np.mgrid[0:H, 0:W]
CENTER_MASK = (((xx-W/2)/(W*.42))**2 + ((yy-H/2)/(H*.42))**2) <= 1
print(f"centre ellipse covers {CENTER_MASK.mean():.1%} of the frame")

par, cams = [], []
for i in range(len(test_idx)):
    cam = gradcam(final_model, X_ALL[test_idx[i]], BEST["head"], K)
    cams.append(cam)
    par.append(float(cam[~CENTER_MASK].sum() / (cam.sum() + 1e-8)))
par = np.array(par)
dfp = pd.DataFrame(dict(true=y_true_t+1, pred=y_pred_t+1, err=np.abs(y_true_t-y_pred_t),
                        conf=predicted_confidence(proba_cal, y_pred_t, 1), peripheral=par))
print(f"\nperipheral attention ratio: mean {par.mean():.3f}  (a uniform map would give {1-CENTER_MASK.mean():.3f})")

fig, ax = plt.subplots(1, 3, figsize=(13, 3.2))
ax[0].hist(par, 25, color="#457b9d"); ax[0].axvline(1-CENTER_MASK.mean(), ls="--", c="#bc4749")
ax[0].text(1-CENTER_MASK.mean(), ax[0].get_ylim()[1]*.9, " uniform", color="#bc4749", fontsize=7)
ax[0].set(xlabel="fraction of CAM mass outside centre ellipse", ylabel="count", title="Peripheral attention")
_present = [c for c in range(K) if (y_true_t == c).sum()]
ax[1].boxplot([par[y_true_t == c] for c in _present])
ax[1].set_xticks(range(1, len(_present)+1), [f"S{c+1}" for c in _present])
ax[1].set(xlabel="true stage", ylabel="peripheral ratio", title="By stage — a trend here mirrors 2.9")
sc = ax[2].scatter(dfp.peripheral, dfp.conf, c=dfp.err, cmap="RdYlGn_r", s=22)
plt.colorbar(sc, ax=ax[2], label="|error| stages")
ax[2].set(xlabel="peripheral ratio", ylabel="calibrated confidence", title="Confident + peripheral = suspect")
savefig("12_shortcut_attention")

sus = dfp[(dfp.peripheral > np.percentile(par, 85)) & (dfp.conf > .6)].sort_values("peripheral", ascending=False)
print(f"\nflagged: {len(sus)} confident predictions driven largely by peripheral pixels")
display(sus.head(8).round(3))
if len(sus):
    s = sus.head(min(5, len(sus))).index.tolist()
    fig, axes = plt.subplots(2, len(s), figsize=(2.1*len(s), 4.4), squeeze=False)
    for k, i in enumerate(s):
        axes[0,k].imshow(X_ALL[test_idx[i]]); axes[0,k].axis("off")
        axes[0,k].set_title(f"S{y_true_t[i]+1}->S{y_pred_t[i]+1}\nperiph {par[i]:.2f}", fontsize=6.5)
        axes[1,k].imshow(overlay(X_ALL[test_idx[i]], cams[i])); axes[1,k].axis("off")
    plt.suptitle("Flagged for manual review — confident, but looking off-scalp", y=1.0)
    savefig("12_flagged")

print("""
Reporting rule: these are candidates for review, not proof of shortcut learning. The honest
statement for the paper is the RATE ("x% of confident test predictions draw >P% of their
attention from outside the head region") plus the reviewed panel, not a claim that the model
"looks at the vertex".""")


## 13 · Leakage ablation — how much does the vendor split inflate results?

The headline experiment. The **same architecture, same hyper-parameters, same seed** is trained
twice:

* **A — vendor protocol:** all 5,783 files, Roboflow's own train/valid/test.
* **B — audited protocol:** §11's locked-test number.

Any gap between them is not model quality. It is duplicate leakage plus the colour shortcut,
measured in the units the paper reports.


In [ ]:
if CFG.RUN_LEAKAGE_ABLATION:
    print("loading all 5,783 vendor files ...")
    X_V = load_images(man.path.tolist()); Y_V = (man.stage.values - 1).astype(np.int32)
    print(f"{X_V.shape}  ({X_V.nbytes/1e6:.0f} MB)")
    v_tr = np.where(man.vendor_split == "train")[0]
    v_va = np.where(man.vendor_split == "valid")[0]
    v_te = np.where(man.vendor_split == "test")[0]

    set_seed(CFG.SEED)
    _Xsave, _Ysave = X_ALL, Y_ALL
    X_ALL, Y_ALL = X_V, Y_V                      # train_model reads the globals
    vm, vh = train_model(BEST["head"], v_tr, v_va, BEST["backbone"], verbose=1, tag="_vendor")
    pv_ = predict(vm, X_V[v_te], BEST["head"])
    X_ALL, Y_ALL = _Xsave, _Ysave

    MV = ordinal_metrics(Y_V[v_te], pv_["y_pred"], pv_["proba"])
    CIV = bootstrap_ci(Y_V[v_te], pv_["y_pred"], pv_["proba"], metrics=PRIMARY + ["accuracy"])
    plot_history(vh, "vendor-split model")
    plot_confusion(Y_V[v_te], pv_["y_pred"], "VENDOR split test set (leaky)", fname="13_confusion_vendor")

    # ---------------------------------------------------------------------------------
    # Can the vendor-trained model be evaluated fairly on OUR locked test set?  No -- and
    # that impossibility is itself the cleanest statement of how total the leakage is.
    # ---------------------------------------------------------------------------------
    _vtr = set(man[man.vendor_split == "train"].group)
    _vva = set(man[man.vendor_split == "valid"].group)
    _tg = ds.loc[test_idx, "group"]
    seen_tr, seen_any = _tg.isin(_vtr).sum(), _tg.isin(_vtr | _vva).sum()
    unseen = np.where(~_tg.isin(_vtr | _vva).values)[0]
    print(f"""
  OVERLAP DIAGNOSTIC — vendor training data vs. our locked test set
     locked-test images the vendor model TRAINED on        : {seen_tr}/{len(test_idx)} ({100*seen_tr/len(test_idx):.1f}%)
     ... trained or validated on                           : {seen_any}/{len(test_idx)} ({100*seen_any/len(test_idx):.1f}%)
     genuinely unseen by the vendor model                  : {len(unseen)}
     stages present among those                            : {sorted(set(Y_ALL[test_idx][unseen] + 1)) if len(unseen) else '-'}

  So the vendor-trained model CANNOT be given a fair test set built from this dataset:
  its training split already covers most of the available photographs. That is the
  finding -- not a number to report alongside the others.""")
    MX = None
    if len(unseen) >= 40 and len(set(Y_ALL[test_idx][unseen])) >= 3:
        pc_ = predict(vm, X_ALL[test_idx][unseen], BEST["head"])
        MX = ordinal_metrics(Y_ALL[test_idx][unseen], pc_["y_pred"], pc_["proba"])
        MX["_n"] = len(unseen)
        print(f"  (a {len(unseen)}-image unseen subset did exist; scored separately, QWK {MX['QWK']:.3f})")
    del vm, X_V; tf.keras.backend.clear_session()
else:
    MV = MX = CIV = None; print("skipped (CFG.RUN_LEAKAGE_ABLATION = False)")


In [ ]:
if MV is not None:
    tbl = pd.DataFrame([
        dict(protocol="A · vendor split (as published)", n_test=len(v_te), **{k: MV[k] for k in PRIMARY+["accuracy","adjacent_acc"]}),
        dict(protocol="B · audited protocol (11)", n_test=len(test_idx), **{k: M_cal[k] for k in PRIMARY+["accuracy","adjacent_acc"]}),
        dict(protocol="— colour-only, vendor split", n_test=len(v_te), **{k: COLOUR_VENDOR.get(k, np.nan) for k in PRIMARY+["accuracy","adjacent_acc"]}),
        dict(protocol="— colour-only, audited split", n_test=len(test_idx), **{k: COLOUR_AUDITED.get(k, np.nan) for k in PRIMARY+["accuracy","adjacent_acc"]}),
    ]).set_index("protocol").round(3)
    print("LEAKAGE ABLATION — identical architecture, identical hyper-parameters, identical seed\n")
    display(tbl)
    tbl.to_csv(f"{CFG.OUT_DIR}/leakage_ablation.csv")

    infl_q = MV["QWK"] - M_cal["QWK"]; infl_a = MV["accuracy"] - M_cal["accuracy"]
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
    lbl = ["vendor split", "audited split"]
    for a, key, t in [(ax[0], "QWK", "quadratic-weighted kappa"), (ax[1], "accuracy", "accuracy")]:
        a.bar(lbl, [MV[key], M_cal[key]], color=["#bc4749", "#2a9d8f"], width=.55)
        a.axhline(COLOUR_VENDOR[key] if key in COLOUR_VENDOR else np.nan, ls=":", c="k", lw=1)
        for i, v in enumerate([MV[key], M_cal[key]]): a.text(i, v+.01, f"{v:.3f}", ha="center", fontsize=9)
        a.set(ylabel=t, ylim=(0, 1.05))
        a.annotate("", xy=(0, MV[key]), xytext=(0, M_cal[key]), arrowprops=dict(arrowstyle="<->", color="#bc4749"))
        a.text(.06, (MV[key]+M_cal[key])/2, f"inflation\n{MV[key]-M_cal[key]:+.3f}", fontsize=7.5, color="#bc4749")
    fig.suptitle("The same model, scored two ways (dotted = colour-only shortcut floor)", y=1.03)
    savefig("13_leakage_ablation")

    print(f"""
================================  HEADLINE RESULT  ================================
  Identical model, identical training recipe. Only the data protocol differs.

     QWK      : {MV['QWK']:.3f} (vendor)  ->  {M_cal['QWK']:.3f} (audited)    inflation {infl_q:+.3f}
     accuracy : {MV['accuracy']:.3f} (vendor)  ->  {M_cal['accuracy']:.3f} (audited)    inflation {infl_a:+.3f}

  For scale: a colour-only logistic regression reaches QWK {COLOUR_VENDOR['QWK']:.3f} on the vendor
  split. Published numbers on this dataset that do not exceed that, on a group-disjoint
  split, have not demonstrated clinical signal.
===================================================================================""")


## 14 · Model card & artefacts

> *"Publish a model card covering intended use, class-level performance, failure cases, and
> reproducibility."* — proposal §7


In [ ]:
_BANNER = ("" if not CFG.QUICK_RUN else
"""> # !! SMOKE TEST -- NOT REPORTABLE !!
> This card was generated with `CFG.QUICK_RUN = True`: a fraction of the epochs, one
> benchmark fold and a reduced bootstrap. Every number below is a pipeline check, not a
> result. Set `QUICK_RUN = False` and re-run before citing anything here.

""")
card = _BANNER + f"""# Model card — ordinal AGA severity grading

**Model.** {FINAL_NAME}. ImageNet-pretrained backbone, two-stage transfer learning
(head-only @ lr {CFG.LR_HEAD}, then fine-tune from `{CFG.FINETUNE_FROM}` @ lr {CFG.LR_FINETUNE},
BatchNorm frozen). Head: `{BEST['head']}`. Input {CFG.IMG_SIZE}x{CFG.IMG_SIZE} RGB, ImageNet
normalisation. Model selection on validation QWK. Confidence temperature-scaled (T = {T:.3f}).

**Intended use.** Research prototype. A quantitative second opinion for dermatologist review and
for methods research on ordinal medical grading.

**Out of scope.** Autonomous diagnosis; treatment recommendation; any clinical deployment; any
use as a substitute for clinical judgement. Not a regulated medical device.

## Data
Source: `hair_norwood_hamilton`, Roboflow Universe (onuralpszr), CC BY 4.0,
https://universe.roboflow.com/onuralpszr/hair_norwood_hamilton

The published dataset contains {len(man):,} files, but they resolve to **{man.group.nunique():,} independent
photographs**; the remainder are colour-augmented copies. The vendor train/valid/test split was
discarded after audit (§2) and replaced with a group-disjoint protocol using one image per group
(n = {len(ds):,}). Split manifest SHA-256: `{LOCK}`.

Audit findings, all reproduced in §2 of the notebook:
1. {leak.loc[0,'pct']:.1f}% of the vendor test set has a near-duplicate in vendor train.
2. Vendor augmentation is confounded with the label (stage 1: 0% augmented; stages 3–7: ~80%).
3. Source collection is confounded with the label (two collections contribute no stage-1 images).
4. Effective class imbalance is {int(ds.stage.value_counts().max()/ds.stage.value_counts().min())}:1, not the ~7:1 the file counts suggest.

## Performance — locked test set, n = {len(test_idx)}, evaluated once
| metric | value | 95% CI |
|---|---|---|
""" + "\n".join(f"| {k} | {M_cal[k]:.3f} | [{CI[k][0]:.3f}, {CI[k][1]:.3f}] |" for k in PRIMARY if k in CI) + f"""
| accuracy | {M_cal['accuracy']:.3f} | [{CI['accuracy'][0]:.3f}, {CI['accuracy'][1]:.3f}] |
| within ±1 stage | {M_cal['adjacent_acc']:.3f} | [{CI['adjacent_acc'][0]:.3f}, {CI['adjacent_acc'][1]:.3f}] |
| ECE (calibrated) | {M_cal['ECE']:.3f} | — |

Per-class performance: see `figures/11_confusion_locked.png` and the table in §11. Per-class
figures for stages with single-digit support are not interpretable in isolation; the pooled
cross-validated results (§10, `benchmark_cv.csv`) are the per-class evidence.

**Reference points.** Human inter-observer reliability on the Hamilton–Norwood scale is
ICC 0.63–0.68 (Guarrera et al., *Int. J. Trichology* 2009). A colour-only logistic regression —
which cannot see hair at all — scores QWK {COLOUR_VENDOR['QWK']:.3f} on the *unaudited* split and
QWK {COLOUR_AUDITED['QWK']:.3f} on the audited one.

## Known limitations
- **Small effective sample.** ~{len(ds):,} independent images; the rarest stage has
  {int(ds.stage.value_counts().min())} images in total. Rare-stage metrics carry very wide CIs.
- **No patient identifiers.** Grouping is by photograph (filename + perceptual hash). If one
  patient was photographed in two separate sessions, those sessions may still straddle the split.
  Patient-level independence is assumed, not verified.
- **Irreversible preprocessing.** Images arrived already stretch-resized to 640×640 with EXIF
  stripped; original framing and aspect ratio cannot be recovered.
- **Single-view.** Mostly top-down/vertex photographs. The Norwood scale is defined on frontal
  *and* vertex patterns; frontal hairline recession is often not visible here.
- **Label subjectivity.** {len(bad)} groups carry internally conflicting labels, all adjacent-stage —
  consistent with the scale's known unreliability. {'These were excluded.' if CFG.EXCLUDE_AMBIGUOUS else 'These were retained.'}
- **Population.** Predominantly one clinical/geographic source; no demographic metadata, so
  fairness across skin tone, age or sex cannot be assessed. Some images appear to be female scalps,
  which the Norwood scale does not cover.
- **Confounds persist.** The source-collection confound (§2.9) exists inside the audited data too;
  aggressive colour augmentation mitigates but does not eliminate it.

## Reproducibility
seed {CFG.SEED} · TF {tf.__version__} · split lock `{LOCK[:32]}...`
Artefacts: `benchmark_cv.csv`, `leakage_ablation.csv`, `split_manifest.csv`,
`audit_manifest.csv`, `final_model.weights.h5`, `figures/`.
"""
open(f"{CFG.OUT_DIR}/MODEL_CARD.md", "w", encoding="utf-8").write(card)
print(card)


In [ ]:
import json
summary = dict(
    dataset=dict(vendor_files=int(len(man)), source_photos=int(man.source_id.nunique()),
                 independent_groups=int(man.group.nunique()), canonical_n=int(len(ds)),
                 class_counts={int(k): int(v) for k, v in ds.stage.value_counts().sort_index().items()}),
    audit=dict(test_leak_pct=float(leak.loc[0, "pct"]), valid_leak_pct=float(leak.loc[1, "pct"]),
               ambiguous_groups=int(len(bad)),
               colour_only_vendor={k: float(v) for k, v in COLOUR_VENDOR.items()},
               colour_only_audited={k: float(v) for k, v in COLOUR_AUDITED.items()}),
    best_model_overall=best_name, final_model=FINAL_NAME,
    temperature=float(T), split_lock=LOCK,
    QUICK_RUN_smoke_test_not_reportable=bool(CFG.QUICK_RUN),
    locked_test={k: float(M_cal[k]) for k in PRIMARY + ["accuracy", "adjacent_acc", "ECE"]},
    residual_similarity=SENS.reset_index().to_dict("records"),
    locked_test_ci={k: [float(a), float(b)] for k, (a, b) in CI.items()},
    leakage_ablation=None if MV is None else dict(
        vendor={k: float(MV[k]) for k in PRIMARY + ["accuracy"]},
        audited={k: float(M_cal[k]) for k in PRIMARY + ["accuracy"]},
        vendor_model_on_unseen_subset=None if MX is None else
            {**{k: float(MX[k]) for k in PRIMARY + ["accuracy"]}, "n": int(MX["_n"])}),
    config={k: (v if isinstance(v, (int, float, str, bool, type(None))) else str(v))
            for k, v in vars(CFG).items()})
json.dump(summary, open(f"{CFG.OUT_DIR}/results_summary.json", "w", encoding="utf-8"), indent=2)

if CFG.DRIVE_BACKUP:
    os.makedirs(CFG.DRIVE_BACKUP, exist_ok=True)
    shutil.copytree(CFG.OUT_DIR, CFG.DRIVE_BACKUP, dirs_exist_ok=True)
    print("copied to", CFG.DRIVE_BACKUP)

print("ARTEFACTS in", CFG.OUT_DIR)
for p in sorted(glob.glob(f"{CFG.OUT_DIR}/**/*", recursive=True)):
    if os.path.isfile(p): print(f"   {os.path.relpath(p, CFG.OUT_DIR):<46} {os.path.getsize(p)/1024:>8.0f} KB")
print("\nDone. If QUICK_RUN was True, set it False and re-run for reportable numbers.")


## 15 · Results dashboard

Everything in one screen: the headline numbers, then every figure the run produced, in reading
order. Nothing is recomputed here — this cell only re-renders what §2–§13 already worked out,
so it is safe to re-run on its own.


In [ ]:
from IPython.display import Image as _Img, Markdown as _Md, HTML as _HTML

def _fmt(v, n=3):
    return "—" if v is None or (isinstance(v, float) and np.isnan(v)) else (
        f"{v:.{n}f}" if isinstance(v, float) else str(v))

_rows = [
    ("Independent images (after audit)", f"{len(ds):,}  (from {len(man):,} vendor files)"),
    ("Rarest stage",                     f"Stage {int(ds.stage.value_counts().idxmin())} — "
                                         f"{int(ds.stage.value_counts().min())} images in total"),
    ("Vendor test-set leakage",          f"{leak.loc[0,'pct']:.1f}% of it sits in vendor train"),
    ("Locked test set",                  f"n = {len(test_idx)}   lock {LOCK[:12]}…"),
    ("", ""),
    ("QWK (locked test)",                f"{_fmt(M_cal['QWK'])}   95% CI [{_fmt(CI['QWK'][0])}, {_fmt(CI['QWK'][1])}]"),
    ("MAE in stages",                    f"{_fmt(M_cal['MAE'],2)}   95% CI [{_fmt(CI['MAE'][0],2)}, {_fmt(CI['MAE'][1],2)}]"),
    ("Macro-F1",                         f"{_fmt(M_cal['macro_f1'])}   95% CI [{_fmt(CI['macro_f1'][0])}, {_fmt(CI['macro_f1'][1])}]"),
    ("Balanced accuracy",                f"{_fmt(M_cal['balanced_acc'])}"),
    ("Exact / within ±1 stage",          f"{M_cal['accuracy']:.1%} / {M_cal['adjacent_acc']:.1%}"),
    ("ECE exact / within ±1",            f"{_fmt(M_cal['ECE'])} / {_fmt(M_cal['ECE_within1'])}"),
    ("", ""),
    ("Colour-only floor (vendor split)", f"QWK {_fmt(COLOUR_VENDOR['QWK'])}  ← a model that cannot see hair"),
    ("Colour-only floor (audited)",      f"QWK {_fmt(COLOUR_AUDITED['QWK'])}"),
    ("Human inter-observer ceiling",     "ICC 0.63–0.68 (Guarrera et al. 2009)"),
]
if MV is not None:
    _rows += [("", ""),
              ("LEAKAGE INFLATION — QWK",      f"{_fmt(MV['QWK'])} vendor → {_fmt(M_cal['QWK'])} audited "
                                               f"({MV['QWK']-M_cal['QWK']:+.3f})"),
              ("LEAKAGE INFLATION — accuracy", f"{MV['accuracy']:.1%} vendor → {M_cal['accuracy']:.1%} audited "
                                               f"({100*(MV['accuracy']-M_cal['accuracy']):+.1f} pts)")]

_banner = ("" if not CFG.QUICK_RUN else
    "<div style='background:#fde2e2;border-left:5px solid #bc4749;padding:9px 13px;margin-bottom:10px'>"
    "<b>QUICK_RUN = True — this is a smoke test.</b> Set it to False and re-run before reporting "
    "any number below.</div>")
_tr = "".join("<tr><td colspan=2 style='height:7px'></td></tr>" if not k else
              f"<tr><td style='padding:3px 18px 3px 0;color:#555'>{k}</td>"
              f"<td style='padding:3px 0;font-weight:600'>{v}</td></tr>" for k, v in _rows)
display(_HTML(f"{_banner}<h3 style='margin:4px 0'>Scorecard — {FINAL_NAME}</h3>"
              f"<table style='font-family:ui-monospace,monospace;font-size:13px'>{_tr}</table>"))

display(_Md("### Cross-validated benchmark (development set)"));  display(show)
display(_Md("### Per-class — locked test set"));                  display(M_cal["_per_class"].round(3))
try:
    display(_Md("### Residual-similarity sensitivity"));          display(SENS)
except NameError:
    pass
if MV is not None:
    display(_Md("### Leakage ablation"));                         display(tbl)


In [ ]:
# ---- every figure, in reading order ----
_CAPTIONS = {
 "02_leakage":               "§2.7  Finding 1 — the vendor split leaks",
 "02_shortcut_augmentation": "§2.8  Finding 2 — the augmentation predicts the label",
 "02_shortcut_source":       "§2.9  Finding 3 — the source collection predicts the label",
 "02_phash_threshold":       "§2.6  Grouping-threshold stability",
 "02_colour_control":        "§2.11 Colour-only control — the shortcut, measured",
 "03_class_distribution":    "§3    True class distribution vs. what the file counts suggest",
 "05_examples":              "§5    Two examples per stage",
 "10_benchmark":             "§10   Benchmark",
 "10_confusions":            "§10   Out-of-fold confusion matrices",
 "10_ordinal_vs_ce":         "§10.1 Ordinal heads vs. softmax",
 "11_confusion_locked":      "§11   Locked-test confusion matrix",
 "11_calibration_errors":    "§11   Calibration, error profile, per-class recall",
 "12_gradcam_panel":         "§12   Grad-CAM — predicted-class posterior",
 "12_shortcut_attention":    "§12.1 Peripheral-attention analysis",
 "12_flagged":               "§12.1 Flagged for manual review",
 "13_confusion_vendor":      "§13   Vendor-split confusion (leaky)",
 "13_leakage_ablation":      "§13   HEADLINE — same model, two protocols",
}
_seen = set()
for _k, _cap in _CAPTIONS.items():
    _p = FIGURES.get(_k, f"{CFG.OUT_DIR}/figures/{_k}.png")
    if os.path.exists(_p):
        display(_Md(f"**{_cap}**")); display(_Img(filename=_p, width=1000)); _seen.add(_k)
for _k, _p in sorted(FIGURES.items()):          # training curves and anything not captioned
    if _k not in _seen and os.path.exists(_p):
        display(_Md(f"**{_k}**")); display(_Img(filename=_p, width=1000))


## 16 · Download bundle

Packs the small artefacts plus every figure into one zip and downloads it.

**Model weights are excluded on purpose.** The five `.h5` checkpoints are ~1.4 GB and nothing in
the analysis needs them — that is what made a naive zip of this folder 1.5 GB. Set
`INCLUDE_WEIGHTS = True` only if you specifically want to ship the trained model.


In [ ]:
INCLUDE_WEIGHTS = False

import zipfile as _zf
SEND = ["results_summary.json", "benchmark_cv.csv", "leakage_ablation.csv",
        "residual_similarity_sensitivity.csv", "test_predictions.csv",
        "split_manifest.csv", "audit_manifest.csv", "MODEL_CARD.md"]

_bundle = f"{CFG.OUT_DIR}/CSE463_results_bundle.zip"
with _zf.ZipFile(_bundle, "w", _zf.ZIP_DEFLATED, compresslevel=9) as z:
    for _f in SEND:
        _p = os.path.join(CFG.OUT_DIR, _f)
        if os.path.exists(_p): z.write(_p, _f)
        else: print(f"   (missing, skipped: {_f})")
    for _p in sorted(glob.glob(f"{CFG.OUT_DIR}/figures/*.png")):
        z.write(_p, f"figures/{os.path.basename(_p)}")
    if INCLUDE_WEIGHTS:
        for _p in sorted(glob.glob(f"{CFG.OUT_DIR}/*.weights.h5")):
            z.write(_p, os.path.basename(_p))

print(f"bundle: {_bundle}   ({os.path.getsize(_bundle)/1e6:.1f} MB)\n")
with _zf.ZipFile(_bundle) as z:
    for _n in z.namelist(): print(f"   {_n}")
_skip = sum(os.path.getsize(_p) for _p in glob.glob(f"{CFG.OUT_DIR}/*.weights.h5"))
if not INCLUDE_WEIGHTS and _skip:
    print(f"\nexcluded {_skip/1e9:.2f} GB of checkpoints (INCLUDE_WEIGHTS = True keeps them)")
if CFG.QUICK_RUN:
    print("\n*** QUICK_RUN was True — this bundle is a smoke test, not results. ***")

try:
    from google.colab import files as _files
    _files.download(_bundle)
except Exception as _e:
    print(f"\n(not running in Colab, or the browser blocked it: {_e})")
    print(f"Download it by hand from the file browser: {_bundle}")


---

## What to write up

**The contribution is the audit.** On this dataset, a careful data-hygiene protocol changes the
headline number by more than any architecture choice will. That is the paper.

Suggested structure:

1. **The dataset is not what it appears to be.** §2 — 5,783 files are ~1,450 photographs; 53% of
   the published test set leaks; the augmentation marker and the source collection both predict
   the label.
2. **Quantify the damage.** §13 — identical model, two protocols, and the gap between them.
   Anchor it with the colour-only control (§2.11): a model that cannot see hair scores QWK 0.715
   on the unaudited split.
3. **Then answer the ordinal question.** §10.1 — on clean data, do CORAL/CORN beat softmax on QWK,
   MAE and the rate of ≥2-stage errors? Report the answer either way; a null result on n≈1,450
   with 19 stage-6 images is informative and honest.
4. **Show where the model looks.** §12 — the panel plus the peripheral-attention *rate*.
5. **State the ceiling.** Human ICC is 0.63–0.68. Position your κ against that, not against 100%.

**What not to claim.** Don't report a single accuracy figure. Don't quote per-class metrics for
stages with single-digit support without their CIs. Don't compare against published numbers on
this dataset's native split without noting that those numbers include the leakage measured in §13.

**Strongest next step if you want more:** get patient identifiers, or a second dataset for
external validation. Every limitation above traces back to ~1,450 photographs with no patient
metadata.

---

### References

1. Cao, Mirjalili & Raschka. Rank consistent ordinal regression for neural networks with
   application to age estimation. *Pattern Recognition Letters* 140:325–331, 2020. (CORAL)
2. Shi, Cao & Raschka. Deep neural networks for rank-consistent ordinal regression based on
   conditional probabilities. *Pattern Analysis and Applications* 26:941–955, 2023. (CORN)
   arXiv:2111.08851
3. Selvaraju et al. Grad-CAM: visual explanations from deep networks via gradient-based
   localization. *ICCV*, 2017.
4. Guarrera et al. Reliability of Hamilton–Norwood classification. *International Journal of
   Trichology* 1(2):120–122, 2009. (ICC 0.63–0.68)
5. He et al. Deep residual learning for image recognition. *CVPR*, 2016. (ResNet-50)
6. Roboflow Universe. `hair_norwood_hamilton` computer vision dataset, CC BY 4.0.
   https://universe.roboflow.com/onuralpszr/hair_norwood_hamilton

*Research prototype. Not a diagnostic device.*
